# A Lei de Coulomb — animação narrada em Manim

Notebook gerado a partir do roteiro *Roteiro Manim — A Lei de Coulomb*. Usa a mesma paleta e o mesmo pipeline (Manim + manim-voiceover com gTTS em pt-BR) do notebook da Lei de Gauss para um fio de cargas.

**Como usar no Colab**

1. Rode a célula de instalação (demora alguns minutos por causa do LaTeX).
2. Rode a célula que grava `coulomb_comum.py` (paleta, voz e utilitários).
3. Rode as cenas na ordem que quiser; cada uma gera um vídeo com narração.
4. A última célula junta todas as cenas renderizadas em `lei_de_coulomb_completo.mp4`.

As células usam `-pql --fps 15` (rápido, 480p), como no notebook da Lei de Gauss. Para a versão final, troque por `-pqh` (1080p, bem mais lento).

Referências: Halliday, *Fundamentos de Física*, cap. 21; notas de Física III, seção 1.3.

In [ ]:
from IPython.display import clear_output

print("Instalando dependências do sistema (LaTeX, FFMPEG, SoX)...")
!sudo apt update -yq
!sudo apt install libcairo2-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science \
    tipa libpango1.0-dev sox dvisvgm -yq

print("Instalando bibliotecas Python (manim, manim-voiceover)...")
!pip install manim "manim-voiceover[gtts]"

clear_output()
print("Ambiente Manim e Voiceover pronto para usar!")

In [ ]:
from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService
print("Manim", __import__("manim").__version__)

## Módulo comum

Paleta herdada do notebook da Lei de Gauss, serviço de voz (gTTS pt-BR com recurso a áudio mudo se a internet falhar) e a classe base `CenaCoulomb`. Mude `USAR_VOZ = False` para testar o tempo das cenas sem narração.

In [ ]:
%%writefile coulomb_comum.py
# =====================================================================
# coulomb_comum.py — paleta, voz e utilitários compartilhados pelas cenas
# Paleta herdada do notebook "Lei de Gauss — fio de cargas"
# =====================================================================
from manim import *
import numpy as np
import hashlib
import subprocess
from pathlib import Path

from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService
from manim_voiceover.services.gtts import GTTSService

# ---------------------------------------------------------------------
# Configuração geral
# ---------------------------------------------------------------------
USAR_VOZ = True          # False = narração silenciosa (só para testar o tempo)
import os
if os.environ.get("COULOMB_SEM_VOZ"):
    USAR_VOZ = False
CARACTERES_POR_SEGUNDO = 13.0

# ---------------------------------------------------------------------
# Paleta (mesmas cores do notebook da Lei de Gauss)
# ---------------------------------------------------------------------
FUNDO = "#181818"
COR_CARGA = RED_E            # carga positiva
COR_CARGA_NEG = BLUE_D       # carga negativa (nova)
COR_FORCA = PURE_YELLOW      # vetores força
COR_UNITARIO = WHITE         # r chapéu e vetores posição
COR_TEXTO = YELLOW           # rótulos e símbolos
COR_COTA = GRAY_B            # cotas, chaves, linhas guia
COR_GAUSS = "#DDDDDD"        # cascas, vidro, fio de torção
SOLARIZED_BLUE = "#2679b4"   # linhas de campo (cena 13)
COR_RESULTADO = BLUE_B       # resultante / resultado final
COR_ERRO = PURE_RED          # erro, cancelamento
COR_PLASTICO = GREEN_E
COR_COBRE = "#B87333"
COR_NEUTRA = GRAY


# ---------------------------------------------------------------------
# Serviços de voz
# ---------------------------------------------------------------------
def _gerar_silencio(destino, duracao):
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "lavfi",
         "-i", "anullsrc=r=24000:cl=mono", "-t", f"{duracao:.2f}",
         "-acodec", "libmp3lame", "-q:a", "9", str(destino)],
        check=True,
    )


class ServicoSilencioso(SpeechService):
    """Gera áudio mudo com duração proporcional ao texto (para testes)."""

    def __init__(self, cps=CARACTERES_POR_SEGUNDO, **kwargs):
        super().__init__(**kwargs)
        self.cps = cps

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        duracao = max(1.2, len(text) / self.cps + 0.4)
        nome = "silencio_" + hashlib.md5(text.encode()).hexdigest()[:12] + ".mp3"
        destino = Path(cache_dir) / nome
        if not destino.exists():
            _gerar_silencio(destino, duracao)
        return {
            "input_text": text,
            "input_data": {"input_text": text, "service": "silencioso"},
            "original_audio": nome,
        }


class VozSegura(GTTSService):
    """gTTS em pt-BR; se a internet falhar, cai para áudio mudo."""

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        try:
            return super().generate_from_text(text, cache_dir=cache_dir, path=path, **kwargs)
        except Exception as erro:  # noqa: BLE001
            logger.warning(f"gTTS falhou ({erro}); usando áudio mudo.")
            return ServicoSilencioso.generate_from_text(self, text, cache_dir, path)

    cps = CARACTERES_POR_SEGUNDO


# ---------------------------------------------------------------------
# Objetos reutilizáveis
# ---------------------------------------------------------------------
def carga_2d(q=1, raio=0.25, cor=None):
    """Carga 2D no padrão do notebook: círculo cheio + sinal branco."""
    if cor is None:
        cor = COR_CARGA if q > 0 else (COR_CARGA_NEG if q < 0 else COR_NEUTRA)
    circ = Circle(radius=raio, color=cor, fill_opacity=1, stroke_width=0)
    if q == 0:
        return VGroup(circ)
    sinal = MathTex("+" if q > 0 else "-", color=WHITE)
    sinal.scale(0.5 * raio / 0.25).set_opacity(0.8)
    return VGroup(circ, sinal)


def carga_3d(q=1, raio=0.25, resolucao=(12, 12), cor=None):
    """Carga 3D no padrão do notebook: Sphere sem brilho."""
    if cor is None:
        cor = COR_CARGA if q > 0 else (COR_CARGA_NEG if q < 0 else COR_NEUTRA)
    esfera = Sphere(radius=raio, resolution=resolucao)
    esfera.set_color(cor)
    esfera.set_sheen(0.0)
    return esfera


def sinal_tex(q=1, escala=0.5, cor=None):
    if cor is None:
        cor = COR_CARGA if q > 0 else COR_CARGA_NEG
    return MathTex("+" if q > 0 else "-", color=cor).scale(escala)


def seta(inicio, fim, cor=COR_FORCA, espessura=5, ponta=0.15, opacidade=1.0):
    """Arrow que não quebra quando o comprimento vai a zero."""
    inicio = np.array(inicio, dtype=float)
    fim = np.array(fim, dtype=float)
    if np.linalg.norm(fim - inicio) < 0.03:
        return VectorizedPoint(inicio)
    s = Arrow(
        inicio, fim, buff=0, color=cor, stroke_width=espessura,
        tip_length=ponta, max_tip_length_to_length_ratio=0.45,
        max_stroke_width_to_length_ratio=12,
    )
    s.set_opacity(opacidade)
    return s


def seta_direcao(origem, direcao, comprimento, maximo=None, **kw):
    """Seta a partir de 'origem' na 'direcao' (normalizada) com comprimento dado."""
    direcao = np.array(direcao, dtype=float)
    n = np.linalg.norm(direcao)
    if n < 1e-9:
        return VectorizedPoint(np.array(origem, dtype=float))
    if maximo is not None:
        comprimento = float(np.clip(comprimento, -maximo, maximo))
    return seta(origem, np.array(origem) + direcao / n * comprimento, **kw)


def caixa_preta(canto=UL, largura=6.0, altura=2.5):
    caixa = Rectangle(width=largura, height=altura, color=BLACK,
                      fill_opacity=0.8, stroke_width=0)
    return caixa.to_corner(canto)


def tremor(amplitude=0.03, frequencia=4.0, fase=0.0, direcao=UP):
    """Updater que faz o objeto 'respirar' sem brigar com outras animações."""
    estado = {"t": 0.0, "ultimo": np.zeros(3)}
    direcao = np.array(direcao, dtype=float)

    def atualizar(m, dt):
        estado["t"] += dt
        novo = amplitude * np.sin(frequencia * estado["t"] + fase) * direcao
        m.shift(novo - estado["ultimo"])
        estado["ultimo"] = novo

    return atualizar


def pontos_fibonacci(n, raio=1.0, centro=ORIGIN):
    pts = []
    phi = np.pi * (3 - np.sqrt(5))
    for i in range(n):
        y = 1 - 2 * (i + 0.5) / n
        r = np.sqrt(1 - y * y)
        th = phi * i
        pts.append(np.array(centro) + raio * np.array([np.cos(th) * r, np.sin(th) * r, y]))
    return pts


def rotacao_z(vetor, angulo):
    c, s = np.cos(angulo), np.sin(angulo)
    x, y, z = vetor
    return np.array([c * x - s * y, s * x + c * y, z])


# ---------------------------------------------------------------------
# Cena base
# ---------------------------------------------------------------------
class CenaCoulomb(VoiceoverScene, ThreeDScene):
    """Base de todas as cenas: fundo, voz e relógio interno."""

    def preparar(self):
        self.camera.background_color = FUNDO
        if USAR_VOZ:
            self.set_speech_service(VozSegura(lang="pt", tld="com.br"))
        else:
            self.set_speech_service(ServicoSilencioso())
        # Relógio: mantém updaters baseados em tempo sempre ativos
        self.relogio = ValueTracker(0)
        self.relogio.add_updater(lambda m, dt: m.increment_value(dt))
        self.add(self.relogio)

    def t(self):
        return self.relogio.get_value()

    def fala(self, texto):
        return self.voiceover(text=texto)

    def fixar(self, *mobs):
        """Fixa no quadro (para cenas 3D) sem desenhar ainda."""
        self.add_fixed_in_frame_mobjects(*mobs)
        self.remove(*mobs)
        return mobs

    def add_fixed_in_frame_mobjects(self, *mobs):
        if not hasattr(self, "_fixos"):
            self._fixos = []
        self._fixos.extend(mobs)
        super().add_fixed_in_frame_mobjects(*mobs)

    def update_mobjects(self, dt):
        # Objetos fixos que mudam de forma (DecimalNumber, Transform) ganham
        # submobjetos novos; re-registramos a família a cada quadro.
        super().update_mobjects(dt)
        if getattr(self, "_fixos", None):
            self.renderer.camera.add_fixed_in_frame_mobjects(*self._fixos)

    def limpar(self, run_time=1.0, manter=()):
        alvos = [m for m in self.mobjects if m is not self.relogio and m not in manter]
        if alvos:
            self.play(*[FadeOut(m) for m in alvos], run_time=run_time)


## Cena 1 — Atração e repulsão (âmbar, vidro e plástico)

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombAbertura
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombAbertura(CenaCoulomb):
    def construct(self):
        self.preparar()
        rng = np.random.default_rng(7)

        # ---------------- 1. Âmbar e lã ----------------
        ambar = Polygon(
            [-0.9, -0.2, 0], [-0.5, 0.45, 0], [0.4, 0.55, 0],
            [1.0, 0.1, 0], [0.6, -0.5, 0], [-0.4, -0.55, 0],
            color=ORANGE, fill_color=ORANGE, fill_opacity=0.85, stroke_width=2,
        ).shift(UP * 0.6)
        la = RoundedRectangle(width=1.6, height=0.7, corner_radius=0.2,
                              color=GRAY_D, fill_opacity=0.9, stroke_width=0)
        la.move_to(ambar.get_top() + UP * 0.15)

        with self.fala("Há mais de dois mil anos, os gregos esfregavam âmbar em lã.") as t:
            self.play(FadeIn(ambar, scale=0.6), FadeIn(la, shift=DOWN), run_time=0.8)
            passadas = [la.animate(rate_func=there_and_back).shift(RIGHT * 1.4) for _ in range(3)]
            self.play(Succession(*passadas), run_time=max(t.duration - 0.8, 1.5))

        # ---------------- 2. Palha atraída ----------------
        palhas = VGroup(*[
            Line(ORIGIN, RIGHT * rng.uniform(0.25, 0.45), color=GOLD_E, stroke_width=4)
            .rotate(rng.uniform(0, PI))
            .move_to([rng.uniform(-3, 3), rng.uniform(-3.3, -2.6), 0])
            for _ in range(12)
        ])
        with self.fala("Depois, o âmbar atraía pedacinhos de palha.") as t:
            self.play(FadeOut(la, shift=UP), FadeIn(palhas, lag_ratio=0.1), run_time=1)
            self.play(ambar.animate.shift(DOWN * 1.6), run_time=0.8)
            alvos = [ambar.point_from_proportion(rng.uniform(0.55, 0.95)) for _ in palhas]
            self.play(
                LaggedStart(*[p.animate.move_to(a).rotate(PI / 3)
                              for p, a in zip(palhas, alvos)], lag_ratio=0.08),
                rate_func=rush_into, run_time=max(t.duration - 1.8, 1.2),
            )

        # ---------------- 3. Título ----------------
        titulo = Text("A Lei de Coulomb", color=COR_TEXTO, font_size=56)
        self.fixar(titulo)
        with self.fala("Hoje sabemos que isso é uma força elétrica.") as t:
            self.play(FadeOut(VGroup(ambar, palhas), shift=UP), Write(titulo), run_time=t.duration * 0.6)
            self.play(titulo.animate.scale(0.5).to_corner(UL), run_time=t.duration * 0.4)

        # =================================================================
        # Montagem 3D: bastão de vidro pendurado
        # =================================================================
        pivo = np.array([0, 0, 1.0])
        suporte = Line([-1.2, 0, 3.0], [1.2, 0, 3.0], color=GRAY_B, stroke_width=10)
        barbante = Line([0, 0, 3.0], pivo, color=COR_GAUSS, stroke_width=2)
        # posição de repouso (fantasma) para comparar o giro
        fantasma = DashedLine(pivo + LEFT * 2, pivo + RIGHT * 2, color=COR_GAUSS,
                              stroke_width=3, dash_length=0.15).set_opacity(0)

        def bastao(cor, opac):
            b = Cylinder(radius=0.17, height=4, direction=RIGHT, resolution=(10, 18))
            b.set_style(fill_color=cor, fill_opacity=opac, stroke_width=0)
            return b

        def sinal_forte(q):
            """Sinal grande, em negrito, sempre virado para a câmera."""
            if q > 0:
                s = MathTex("+", color=COR_CARGA).scale(1.15)
                s.set_stroke(COR_CARGA, width=4)
            else:
                s = MathTex("-", color=WHITE).scale(1.5)
                s.set_stroke(WHITE, width=5)
            return s

        offsets = [np.array([x, 0, 0]) for x in np.linspace(-1.6, 1.6, 8)]
        ACIMA = OUT * 0.3  # sinais ficam por cima do bastão

        vidro1 = bastao(COR_GAUSS, 0.5).move_to(pivo)
        ang = ValueTracker(0.0)       # ângulo comandado
        balanco = ValueTracker(0.05)  # leve oscilação "viva"

        def ang_total():
            return ang.get_value() + balanco.get_value() * np.sin(1.2 * self.t())

        vidro1.ang_atual = 0.0

        def girar(m):
            alvo = ang_total()
            m.rotate(alvo - m.ang_atual, axis=OUT, about_point=pivo)
            m.ang_atual = alvo

        sinais1 = VGroup(*[sinal_forte(+1) for _ in offsets])

        def seguir1(g):
            for s, o in zip(g, offsets):
                s.move_to(pivo + rotacao_z(o, ang_total()) + ACIMA)

        seguir1(sinais1)
        self.add_fixed_orientation_mobjects(*sinais1)
        self.remove(*sinais1)

        # ---------------- 4. Bastão pendurado ----------------
        with self.fala("Vamos repetir com dois bastões de vidro.") as t:
            self.move_camera(phi=55 * DEGREES, theta=-75 * DEGREES, zoom=1.3, run_time=t.duration * 0.5)
            self.play(FadeIn(VGroup(suporte, barbante), shift=IN), FadeIn(vidro1, shift=IN),
                      run_time=t.duration * 0.4)
            vidro1.add_updater(girar)
            self.begin_ambient_camera_rotation(rate=0.02)

        # ---------------- 5. Esfregar com seda ----------------
        P_FORA = np.array([1.5, -4.6, 0.55])
        P_PERTO = np.array([1.5, -1.5, 0.55])
        vidro2 = bastao(COR_GAUSS, 0.5).move_to(P_FORA)
        sinais2 = VGroup(*[sinal_forte(+1).move_to(P_FORA + o + ACIMA) for o in offsets])
        self.add_fixed_orientation_mobjects(*sinais2)
        self.remove(*sinais2)
        grupo2 = VGroup(vidro2, sinais2)
        seda = Square(0.55, color=WHITE, fill_opacity=0.4, stroke_width=0).rotate(90 * DEGREES, RIGHT)
        seda.move_to(P_FORA + LEFT * 1.9 + OUT * 0.35)

        with self.fala("Esfregamos cada um com seda.") as t:
            self.play(FadeIn(vidro2, shift=UP), FadeIn(seda), run_time=0.8)
            sinais1.add_updater(seguir1)
            self.add(sinais1)
            self.play(seda.animate(rate_func=there_and_back).shift(RIGHT * 3.8),
                      LaggedStart(*[GrowFromCenter(s) for s in sinais2], lag_ratio=0.15),
                      LaggedStart(*[GrowFromCenter(s) for s in sinais1], lag_ratio=0.15),
                      run_time=max(t.duration - 1.1, 1.5))
            self.play(FadeOut(seda), run_time=0.3)

        # ---------------- Setas, arco e rótulo ----------------
        mostrar = ValueTracker(0.0)
        modo = {"s": +1}            # +1 repulsão, -1 atração
        outro = {"m": vidro2}       # bastão que se aproxima

        def seta_pendurado():
            p = pivo + rotacao_z(RIGHT * 1.3, ang_total()) + ACIMA
            d = rotacao_z(UP, ang_total()) * modo["s"]
            return seta_direcao(p, d, 1.4 * mostrar.get_value(), espessura=12, ponta=0.32)

        def seta_outro():
            c = outro["m"].get_center() + ACIMA
            return seta_direcao(c, DOWN * modo["s"], 1.4 * mostrar.get_value(), espessura=12, ponta=0.32)

        def arco_giro():
            a = ang_total()
            if abs(a) < 0.03 or mostrar.get_value() < 0.01:
                return VectorizedPoint(pivo)
            arco = Arc(radius=2.25, start_angle=0, angle=a, arc_center=pivo,
                       color=COR_TEXTO, stroke_width=5).set_opacity(mostrar.get_value())
            return arco.add_tip(tip_length=0.25)

        f_pend = always_redraw(seta_pendurado)
        f_outro = always_redraw(seta_outro)
        arco = always_redraw(arco_giro)
        self.add(f_pend, f_outro, arco)

        rot_rep = Text("repulsão", font_size=40, color=COR_TEXTO).to_edge(UP, buff=0.35)
        rot_atr = Text("atração", font_size=40, color=COR_TEXTO).to_edge(UP, buff=0.35)
        self.fixar(rot_rep, rot_atr)

        # ---------------- 6. Repulsão ----------------
        with self.fala("Ao aproximá-los, o bastão pendurado foge.") as t:
            self.stop_ambient_camera_rotation()
            self.add(fantasma)
            self.play(grupo2.animate.move_to(P_PERTO + ACIMA / 2), balanco.animate.set_value(0),
                      fantasma.animate.set_opacity(0.5), run_time=t.duration * 0.35)
            self.play(ang.animate.set_value(55 * DEGREES), mostrar.animate.set_value(1),
                      FadeIn(rot_rep, shift=DOWN), run_time=t.duration * 0.65,
                      rate_func=rate_functions.ease_out_back)
        # o bastão "recua e volta": a repulsão continua visível
        self.play(grupo2.animate(rate_func=there_and_back).shift(UP * 0.4),
                  ang.animate(rate_func=there_and_back).set_value(68 * DEGREES), run_time=1.4)

        # ---------------- 7. Bastão negativo (azul) ----------------
        plastico = bastao(COR_CARGA_NEG, 1.0).move_to(P_FORA + RIGHT * 7)
        sinais_p = VGroup(*[sinal_forte(-1).move_to(P_FORA + o + ACIMA) for o in offsets])
        self.add_fixed_orientation_mobjects(*sinais_p)
        self.remove(*sinais_p)
        grupo_p = VGroup(plastico, sinais_p)
        la2 = RoundedRectangle(width=0.7, height=0.5, corner_radius=0.1, color=GRAY_D,
                               fill_opacity=0.95, stroke_width=0).rotate(90 * DEGREES, RIGHT)

        with self.fala("Agora, um bastão de plástico esfregado com lã.") as t:
            self.play(grupo2.animate.shift(LEFT * 8), FadeOut(rot_rep),
                      mostrar.animate.set_value(0), ang.animate.set_value(0),
                      plastico.animate.move_to(P_FORA), run_time=t.duration * 0.4)
            self.remove(grupo2)
            outro["m"] = plastico
            la2.move_to(P_FORA + LEFT * 1.9 + OUT * 0.4)
            self.add(la2)
            self.play(la2.animate(rate_func=there_and_back).shift(RIGHT * 3.8),
                      LaggedStart(*[GrowFromCenter(s) for s in sinais_p], lag_ratio=0.12),
                      run_time=t.duration * 0.6)
            self.remove(la2)

        # ---------------- 8. Atração ----------------
        with self.fala("Desta vez, os bastões se atraem.") as t:
            modo["s"] = -1
            self.play(grupo_p.animate.move_to(P_PERTO + DOWN * 0.15 + ACIMA / 2), run_time=t.duration * 0.35)
            self.play(ang.animate.set_value(-28 * DEGREES), mostrar.animate.set_value(1),
                      FadeIn(rot_atr, shift=DOWN), run_time=t.duration * 0.65,
                      rate_func=rate_functions.ease_out_back)
        self.play(ang.animate(rate_func=there_and_back).set_value(-33 * DEGREES),
                  grupo_p.animate(rate_func=there_and_back).shift(UP * 0.15), run_time=1.2)

        # ---------------- 9. Regra ----------------
        caixa = Rectangle(width=10, height=0.9, color=BLACK, fill_opacity=0.8, stroke_width=0)
        caixa.to_edge(DOWN, buff=0.4)
        regra = Text("Mesmo sinal se repele. Sinais opostos se atraem.",
                     font_size=30, color=COR_TEXTO).move_to(caixa)
        self.fixar(caixa, regra)
        with self.fala("Mesmo sinal se repele. Sinais opostos se atraem.") as t:
            self.play(FadeIn(caixa), Write(regra), run_time=1.5)
            self.play(LaggedStart(Indicate(sinais1, color=WHITE, scale_factor=1.4),
                                  Indicate(sinais_p, color=COR_TEXTO, scale_factor=1.4),
                                  lag_ratio=0.5), run_time=1.5)
            self.begin_ambient_camera_rotation(rate=0.06)
            self.wait(max(t.duration - 3, 0.5))
        self.wait(1)

## Cena 2 — Dois tipos de carga, condutores e indução

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombCondutores
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombCondutores(CenaCoulomb):
    def construct(self):
        self.preparar()
        rng = np.random.default_rng(3)

        # ---------------- 1. Positiva e negativa ----------------
        giro = ValueTracker(0)
        giro.add_updater(lambda m, dt: m.increment_value(0.9 * dt))
        cp = carga_2d(+1, 0.45)
        cn = carga_2d(-1, 0.45)
        cp.add_updater(lambda m: m.move_to(1.4 * np.array([np.cos(giro.get_value()), np.sin(giro.get_value()), 0])))
        cn.add_updater(lambda m: m.move_to(-1.4 * np.array([np.cos(giro.get_value()), np.sin(giro.get_value()), 0])))
        rot_p = Text("positiva", font_size=28, color=COR_TEXTO)
        rot_n = Text("negativa", font_size=28, color=COR_TEXTO)
        rot_p.add_updater(lambda m: m.next_to(cp, UP, buff=0.2))
        rot_n.add_updater(lambda m: m.next_to(cn, UP, buff=0.2))

        with self.fala("Franklin chamou as cargas de positiva e negativa.") as t:
            self.add(giro)
            self.play(GrowFromCenter(cp), GrowFromCenter(cn), run_time=1)
            self.play(Write(rot_p), Write(rot_n), run_time=max(t.duration - 1, 1))

        # ---------------- 2. Objeto neutro ----------------
        objeto = RoundedRectangle(width=6, height=3, corner_radius=0.3, color=COR_GAUSS,
                                  stroke_width=3, fill_opacity=0.05)
        pares = VGroup()
        for i in range(5):
            for j in range(4):
                centro = np.array([-2.2 + 1.1 * i, -1.05 + 0.7 * j, 0])
                par = VGroup(carga_2d(+1, 0.16).move_to(centro + LEFT * 0.17),
                             carga_2d(-1, 0.16).move_to(centro + RIGHT * 0.17))
                pares.add(par)

        with self.fala("Um objeto comum tem as duas em igual quantidade.") as t:
            self.play(FadeOut(VGroup(cp, cn, rot_p, rot_n)), Create(objeto), run_time=1)
            self.play(LaggedStart(*[FadeIn(p, scale=0.5) for p in pares], lag_ratio=0.05),
                      run_time=max(t.duration - 1, 1))
            for k, p in enumerate(pares):
                p.add_updater(tremor(0.03, 5, fase=k * 0.7, direcao=rng.normal(size=3) * [1, 1, 0]))

        # ---------------- 3. Neutro ----------------
        q_total = MathTex(r"q_{\text{total}} = 0", color=COR_TEXTO).next_to(objeto, UP, buff=0.4)
        pontos = VGroup(*[Dot(p.get_center(), radius=0.12, color=COR_NEUTRA) for p in pares])
        with self.fala("Dizemos que ele está eletricamente neutro.") as t:
            for p in pares:
                p.clear_updaters()
            self.play(*[Transform(p, d) for p, d in zip(pares, pontos)], Write(q_total),
                      run_time=t.duration * 0.6)
            for k, p in enumerate(pares):
                p.add_updater(tremor(0.025, 5, fase=k, direcao=UP))
            self.play(Indicate(q_total, scale_factor=1.3), run_time=t.duration * 0.4)

        # ---------------- 4. Condutor: cobre ----------------
        centro_cobre = np.array([0, 1.4, 0])
        cobre = RoundedRectangle(width=7, height=1.2, corner_radius=0.2, color=COR_COBRE,
                                 fill_opacity=0.35, stroke_width=3).move_to(centro_cobre)
        rot_cobre = Text("cobre (condutor)", font_size=24, color=COR_COBRE).next_to(cobre, UP, buff=0.15)
        ions = VGroup(*[carga_2d(+1, 0.12).move_to(centro_cobre + [x, y, 0])
                        for x in np.linspace(-3.0, 3.0, 10) for y in (-0.3, 0.3)])
        eletrons = VGroup(*[carga_2d(-1, 0.09).move_to(centro_cobre + [x, y, 0])
                            for x, y in zip(np.linspace(-3.1, 3.1, 20), rng.uniform(-0.45, 0.45, 20))])
        lim = {"xmin": -3.3, "xmax": 3.3}

        def passeio(m, dt):
            if not hasattr(m, "v"):
                m.v = rng.normal(size=3) * [1, 1, 0]
            m.v += rng.normal(size=3) * [1, 1, 0] * dt * 6
            m.v *= 0.98
            m.v = np.clip(m.v, -1.5, 1.5)
            p = m.get_center() + m.v * dt
            rel = p - centro_cobre
            if rel[0] < lim["xmin"] or rel[0] > lim["xmax"]:
                m.v[0] *= -1
            if abs(rel[1]) > 0.45:
                m.v[1] *= -1
            m.shift(m.v * dt)

        with self.fala("No cobre, os elétrons se movem com facilidade.") as t:
            self.play(FadeOut(VGroup(objeto, pares, q_total)), run_time=0.7)
            self.play(FadeIn(cobre), Write(rot_cobre), FadeIn(ions), FadeIn(eletrons),
                      run_time=1.2)
            for e in eletrons:
                e.add_updater(passeio)

        # ---------------- 5. Isolante: vidro ----------------
        centro_vidro = np.array([0, -1.8, 0])
        vidro = RoundedRectangle(width=7, height=1.2, corner_radius=0.2, color=COR_GAUSS,
                                 fill_opacity=0.15, stroke_width=3).move_to(centro_vidro)
        rot_vidro = Text("vidro (isolante)", font_size=24, color=COR_GAUSS).next_to(vidro, DOWN, buff=0.15)
        cargas_vidro = VGroup()
        for k, x in enumerate(np.linspace(-3.0, 3.0, 10)):
            for y, q in ((-0.25, +1), (0.25, -1)):
                c = carga_2d(q, 0.11).move_to(centro_vidro + [x + 0.1 * q, y, 0])
                c.add_updater(tremor(0.02, 7, fase=k + y, direcao=RIGHT))
                cargas_vidro.add(c)

        with self.fala("No vidro, um isolante, as cargas ficam presas.") as t:
            self.play(FadeIn(vidro), Write(rot_vidro), FadeIn(cargas_vidro, lag_ratio=0.05),
                      run_time=t.duration)

        # ---------------- 6. Plástico negativo ----------------
        plastico = VGroup(
            RoundedRectangle(width=2.6, height=0.45, corner_radius=0.15, color=COR_PLASTICO,
                             fill_opacity=1, stroke_width=0),
        )
        plastico.add(*[carga_2d(-1, 0.1).move_to(plastico[0].get_center() + RIGHT * x)
                       for x in np.linspace(-1.0, 1.0, 6)])
        plastico.move_to([-12, 0.0, 0])
        with self.fala("Aproximamos um plástico negativo do cobre.") as t:
            self.play(FadeOut(VGroup(vidro, rot_vidro, cargas_vidro)),
                      plastico.animate.move_to([-3.4, 0.1, 0]), run_time=t.duration)

        # ---------------- 7. Elétrons fogem ----------------
        with self.fala("Os elétrons livres fogem para a ponta oposta.") as t:
            for e in eletrons:
                e.clear_updaters()
            destinos = [centro_cobre + [rng.uniform(1.6, 3.2), rng.uniform(-0.45, 0.45), 0] for _ in eletrons]
            self.play(LaggedStart(*[e.animate.move_to(d) for e, d in zip(eletrons, destinos)],
                                  lag_ratio=0.04), run_time=t.duration * 0.6)
            lim["xmin"] = 1.5
            for e in eletrons:
                e.v = np.zeros(3)
                e.add_updater(passeio)
            expostos = VGroup(*[i for i in ions if i.get_center()[0] < -1.5])
            self.play(Indicate(expostos, color=RED_A, scale_factor=1.4), run_time=t.duration * 0.4)

        # ---------------- 8. Atração vence ----------------
        f_atr = seta(cobre.get_left() + RIGHT * 0.5 + DOWN * 0.1, cobre.get_left() + RIGHT * 0.5 + DOWN * 1.3)
        f_rep = seta(cobre.get_right() + LEFT * 0.5 + DOWN * 0.1, cobre.get_right() + LEFT * 0.5 + UP * 0.4)
        barra = VGroup(cobre, ions, eletrons, f_atr, f_rep)
        with self.fala("A ponta positiva está mais perto, e a atração vence.") as t:
            self.play(GrowArrow(f_atr), GrowArrow(f_rep), run_time=t.duration * 0.4)
            for e in eletrons:
                e.clear_updaters()
            self.play(Rotate(barra, angle=12 * DEGREES, about_point=centro_cobre),
                      FadeOut(rot_cobre), run_time=t.duration * 0.6, rate_func=smooth)
            for k, e in enumerate(eletrons):
                e.add_updater(tremor(0.04, 6, fase=k, direcao=RIGHT))

        # ---------------- 9. Só elétrons se movem ----------------
        caixa = Rectangle(width=11, height=0.9, color=BLACK, fill_opacity=0.8, stroke_width=0).to_edge(DOWN, buff=0.4)
        frase = Text("Só os elétrons se movem. Os íons ficam no lugar.", font_size=30,
                     color=COR_TEXTO).move_to(caixa)
        with self.fala("Só os elétrons se movem. Os íons ficam no lugar.") as t:
            self.play(FadeIn(caixa), Write(frase), run_time=1.2)
            self.play(Wiggle(eletrons), run_time=max(t.duration / 2 - 0.6, 0.8))
            self.play(Indicate(ions, color=RED_A), run_time=max(t.duration / 2 - 0.6, 0.8))
        self.wait(0.5)

## Cena 3 — A balança de torção de Coulomb

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombBalanca
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombBalanca(CenaCoulomb):
    def construct(self):
        self.preparar()
        self.set_camera_orientation(phi=65 * DEGREES, theta=-45 * DEGREES, zoom=1.1)
        self.begin_ambient_camera_rotation(rate=0.03)

        z_haste = 0.3
        pivo = np.array([0, 0, z_haste])
        topo = np.array([0, 0, 3.6])
        r_haste = 1.6

        # ---------------- Estrutura da balança ----------------
        cilindro = Cylinder(radius=2.2, height=2.2, direction=OUT, resolution=(4, 32))
        cilindro.set_style(fill_color=COR_GAUSS, fill_opacity=0.12, stroke_color=COR_GAUSS,
                           stroke_width=0.5, stroke_opacity=0.25).move_to([0, 0, 0.0])
        tubo = Cylinder(radius=0.22, height=2.5, direction=OUT, resolution=(2, 16))
        tubo.set_style(fill_color=COR_GAUSS, fill_opacity=0.12, stroke_width=0).move_to([0, 0, 2.35])

        torcao = ValueTracker(0.0)   # ângulo de torção do fio (rad)
        espessura_fio = ValueTracker(2.0)

        def fio():
            voltas = 3 + 3 * torcao.get_value()
            return ParametricFunction(
                lambda s: np.array([0.035 * np.cos(2 * PI * voltas * s),
                                    0.035 * np.sin(2 * PI * voltas * s),
                                    topo[2] + (z_haste - topo[2]) * s]),
                t_range=[0, 1, 0.01], color=WHITE, stroke_width=espessura_fio.get_value(),
            )

        fio_torcao = always_redraw(fio)

        # Escala angular
        escala = VGroup(Circle(radius=2.0, color=COR_COTA, stroke_width=2, stroke_opacity=0.6))
        for k in range(36):
            a = k * 10 * DEGREES
            d = np.array([np.cos(a), np.sin(a), 0])
            escala.add(Line(1.9 * d, 2.0 * d, color=COR_COTA, stroke_width=2))
        escala.move_to(pivo)

        # Haste com esfera g e contrapeso
        haste_linha = Line(pivo + LEFT * r_haste, pivo + RIGHT * r_haste, color=GRAY_B, stroke_width=6)
        esfera_g = carga_3d(0, 0.17, cor=GRAY_A).move_to(pivo + RIGHT * r_haste)
        contrapeso = Cylinder(radius=0.18, height=0.05, direction=RIGHT, resolution=(2, 12))
        contrapeso.set_style(fill_color=GRAY_B, fill_opacity=1, stroke_width=0).move_to(pivo + LEFT * r_haste)
        haste = VGroup(haste_linha, esfera_g, contrapeso)
        haste.ang_atual = 0.0
        ang = ValueTracker(0.0)

        def girar_haste(m):
            alvo = ang.get_value()
            m.rotate(alvo - m.ang_atual, axis=OUT, about_point=pivo)
            m.ang_atual = alvo

        # Oscilação livre amortecida (controlada por um relógio próprio)
        osc = {"t": 0.0, "amp": 0.3, "w": 1.6, "ativo": False}

        def oscilar(m, dt):
            if osc["ativo"]:
                osc["t"] += dt
                m.set_value(osc["amp"] * np.sin(osc["w"] * osc["t"]) * np.exp(-osc["t"] / 6))

        ang.add_updater(oscilar)
        torcao.add_updater(lambda m: m.set_value(ang.get_value()))
        self.add(ang, torcao)

        # ---------------- 1. Montagem ----------------
        ano = Text("1785", font_size=48, color=COR_TEXTO).to_corner(UR)
        self.fixar(ano)
        with self.fala("Em 1785, Coulomb quis medir essa força.") as t:
            self.play(Create(cilindro), Create(tubo), FadeIn(ano, shift=DOWN), run_time=t.duration * 0.6)
            self.play(Create(fio_torcao), Create(escala), FadeOut(ano), run_time=t.duration * 0.4)

        # ---------------- 2. Haste oscilando ----------------
        with self.fala("Seu instrumento foi a balança de torção.") as t:
            self.play(GrowFromCenter(haste), run_time=1)
            haste.add_updater(girar_haste)
            osc.update(t=0.0, amp=0.35, w=1.6, ativo=True)
            self.wait(max(t.duration - 1, 0.5))

        # ---------------- 3. Torque no fio ----------------
        seta_tau = Arc(radius=0.35, start_angle=0, angle=1.5 * PI, color=COR_TEXTO, stroke_width=4)
        seta_tau.add_tip(tip_length=0.15).move_to([0, 0, 2.0])
        rotulo_tau = MathTex(r"\tau", color=COR_TEXTO).scale(0.9)
        with self.fala("Um fio fino de metal, quando torcido, reage com um torque.") as t:
            self.stop_ambient_camera_rotation()
            self.move_camera(zoom=1.7, frame_center=[0, 0, 1.6], run_time=t.duration * 0.5,
                             added_anims=[Create(seta_tau)])
            self.add_fixed_orientation_mobjects(rotulo_tau)
            rotulo_tau.move_to([0.6, 0, 2.3])
            self.play(Write(rotulo_tau), Rotate(seta_tau, angle=PI, axis=OUT, about_point=[0, 0, 2.0]),
                      run_time=t.duration * 0.5)

        # ---------------- 4. Torque proporcional ao ângulo ----------------
        caixa = caixa_preta(UL, 5.2, 1.6)
        eq_tau = MathTex(r"\tau", "=", "K", r"\,\frac{d^4}{l}", r"\,\varphi", font_size=48).move_to(caixa)
        self.fixar(caixa, eq_tau)
        with self.fala("O torque é proporcional ao ângulo de torção.") as t:
            osc["ativo"] = False
            self.play(FadeIn(caixa), Write(eq_tau), ang.animate.set_value(0),
                      run_time=t.duration * 0.35)
            self.play(ang.animate(rate_func=there_and_back).set_value(40 * DEGREES),
                      Indicate(eq_tau[0], color=COR_TEXTO), Indicate(eq_tau[4], color=COR_TEXTO),
                      seta_tau.animate(rate_func=there_and_back).scale(1.5),
                      run_time=t.duration * 0.65)

        # ---------------- 5. Diâmetro à quarta ----------------
        with self.fala("E cresce com a quarta potência do diâmetro do fio.") as t:
            osc.update(t=0.0, amp=0.3, w=3.2, ativo=True)
            self.play(espessura_fio.animate.set_value(7), Indicate(eq_tau[3], color=COR_TEXTO),
                      run_time=t.duration * 0.6)
            self.play(espessura_fio.animate.set_value(2.5), run_time=t.duration * 0.4)

        # ---------------- 6. Eletrizar g ----------------
        mais = VGroup(*[Dot3D(pivo + RIGHT * r_haste + np.array([0.9 * np.cos(a), 0.9 * np.sin(a), 0.5]),
                              radius=0.04, color=COR_CARGA) for a in np.linspace(0, TAU, 8, endpoint=False)])
        with self.fala("Agora eletrizamos a esfera da haste.") as t:
            osc["ativo"] = False
            self.move_camera(zoom=1.15, frame_center=[0, 0, 0.6], run_time=t.duration * 0.4,
                             added_anims=[ang.animate.set_value(0), FadeOut(seta_tau), FadeOut(rotulo_tau)])
            self.begin_ambient_camera_rotation(rate=0.03)
            self.play(FadeIn(mais), run_time=0.3)
            self.play(LaggedStart(*[d.animate.move_to(esfera_g.get_center()).set_opacity(0) for d in mais],
                                  lag_ratio=0.1),
                      esfera_g.animate.set_color(COR_CARGA), run_time=max(t.duration * 0.6 - 0.3, 0.8))
            self.remove(mais)

        # ---------------- 7. Esfera J ----------------
        a_j = -18 * DEGREES
        p_j = pivo + r_haste * np.array([np.cos(a_j), np.sin(a_j), 0])
        esfera_j = carga_3d(+1, 0.17).move_to(p_j + OUT * 2.8)
        haste_j = Line(p_j + OUT * 3.3, p_j + OUT * 2.8, color=GRAY_B, stroke_width=4)
        grupo_j = VGroup(esfera_j, haste_j)
        with self.fala("E aproximamos outra esfera carregada.") as t:
            self.play(FadeIn(grupo_j), run_time=0.5)
            self.play(grupo_j.animate.shift(IN * 2.8), run_time=max(t.duration - 0.5, 1))

        # ---------------- 8. Repulsão e equilíbrio ----------------
        leitura = VGroup(MathTex(r"\varphi =", color=COR_TEXTO),
                         DecimalNumber(0, num_decimal_places=1, unit=r"^\circ", color=COR_TEXTO))
        leitura.arrange(RIGHT).to_corner(UR)
        leitura[1].add_updater(lambda m: m.set_value(ang.get_value() / DEGREES))
        self.fixar(leitura)
        with self.fala("A repulsão gira a haste até o fio equilibrar o torque.") as t:
            self.play(FadeIn(leitura), run_time=0.5)
            self.play(ang.animate.set_value(32 * DEGREES), run_time=max(t.duration - 0.5, 1),
                      rate_func=rate_functions.ease_out_back)

        # ---------------- 9. Medindo a força ----------------
        def seta_forca():
            a = ang.get_value()
            pos = pivo + r_haste * np.array([np.cos(a), np.sin(a), 0])
            tang = np.array([-np.sin(a), np.cos(a), 0])
            return seta_direcao(pos, tang, 1.8 * a, espessura=7, ponta=0.22)

        def arco_angulo():
            a = ang.get_value()
            if a < 0.02:
                return VectorizedPoint(pivo)
            return Arc(radius=2.0, start_angle=0, angle=a, color=COR_TEXTO, stroke_width=6).shift(pivo)

        f_g = always_redraw(seta_forca)
        arco = always_redraw(arco_angulo)
        with self.fala("Medindo o ângulo, Coulomb media a força.") as t:
            self.add(f_g, arco)
            self.play(ang.animate(rate_func=there_and_back).set_value(12 * DEGREES), run_time=t.duration * 0.5)
            self.play(Indicate(leitura, color=COR_TEXTO), run_time=t.duration * 0.5)

        # ---------------- 10. Como comparar cargas? ----------------
        balanca = VGroup(cilindro, tubo, escala, haste, grupo_j)
        # esferas penduradas desenhadas no quadro (2D fixo)
        def pendurada(x, cor, rotulo):
            fio2 = Line([x, 2.3, 0], [x, 0.8, 0], color=COR_GAUSS, stroke_width=2)
            bola = Circle(radius=0.32, color=cor, fill_opacity=1, stroke_width=0).move_to([x, 0.5, 0])
            txt = MathTex(rotulo, color=COR_TEXTO, font_size=34).next_to(bola, DOWN, buff=0.15)
            return VGroup(fio2, bola, txt)

        suporte2 = Line([1.6, 2.3, 0], [6.0, 2.3, 0], color=GRAY_B, stroke_width=6)
        a_esf = pendurada(3.0, COR_CARGA, "q_0")
        b_esf = pendurada(4.6, COR_NEUTRA, "0")
        self.fixar(suporte2, a_esf, b_esf)
        with self.fala("Mas como comparar quantidades de carga?") as t:
            self.play(FadeOut(VGroup(f_g, arco, leitura, caixa, eq_tau, fio_torcao)),
                      balanca.animate.shift(LEFT * 4.6 + OUT * 0.3).set_opacity(0.15),
                      run_time=t.duration * 0.5)
            self.play(FadeIn(suporte2, shift=DOWN), FadeIn(a_esf, shift=LEFT), FadeIn(b_esf, shift=LEFT),
                      run_time=t.duration * 0.5)

        # ---------------- 11. Contato ----------------
        with self.fala("Encostamos uma esfera carregada numa idêntica e neutra.") as t:
            self.play(Rotate(a_esf, angle=15.5 * DEGREES, about_point=[3.0, 2.3, 0]),
                      Rotate(b_esf, angle=-15.5 * DEGREES, about_point=[4.6, 2.3, 0]),
                      run_time=t.duration * 0.5)
            self.play(b_esf[1].animate.set_color(interpolate_color(COR_CARGA, WHITE, 0.25)),
                      a_esf[1].animate.set_color(interpolate_color(COR_CARGA, WHITE, 0.25)),
                      run_time=t.duration * 0.5)

        # ---------------- 12. Metade para cada ----------------
        meio_a = MathTex(r"q_0/2", color=COR_TEXTO, font_size=34)
        meio_b = MathTex(r"q_0/2", color=COR_TEXTO, font_size=34)
        barras = VGroup()
        valores = [1, 1 / 2, 1 / 4, 1 / 8]
        for k, v in enumerate(valores):
            barras.add(Rectangle(width=0.45, height=1.7 * v, color=COR_CARGA, fill_opacity=0.8,
                                 stroke_width=0))
        barras.arrange(RIGHT, buff=0.25, aligned_edge=DOWN).move_to([3.8, -3.3, 0], aligned_edge=DOWN)
        rot_barras = VGroup(*[MathTex(s, font_size=26, color=COR_TEXTO).next_to(b, DOWN, buff=0.1)
                              for s, b in zip([r"q_0", r"q_0/2", r"q_0/4", r"q_0/8"], barras)])
        self.fixar(meio_a, meio_b, barras, rot_barras)
        with self.fala("Por simetria, cada uma fica com q zero sobre dois.") as t:
            self.play(Rotate(a_esf, angle=-15.5 * DEGREES, about_point=[3.0, 2.3, 0]),
                      Rotate(b_esf, angle=15.5 * DEGREES, about_point=[4.6, 2.3, 0]),
                      run_time=t.duration * 0.3)
            meio_a.move_to(a_esf[2]); meio_b.move_to(b_esf[2])
            self.play(Transform(a_esf[2], meio_a), Transform(b_esf[2], meio_b),
                      GrowFromEdge(barras[0], DOWN), GrowFromEdge(barras[1], DOWN),
                      FadeIn(rot_barras[:2]), run_time=t.duration * 0.3)
            # cadeia: a esfera B encosta em novas esferas neutras
            self.play(Succession(
                AnimationGroup(Indicate(b_esf[1], scale_factor=1.3), GrowFromEdge(barras[2], DOWN),
                               FadeIn(rot_barras[2])),
                AnimationGroup(Indicate(b_esf[1], scale_factor=1.3), GrowFromEdge(barras[3], DOWN),
                               FadeIn(rot_barras[3])),
            ), run_time=t.duration * 0.4)

        # ---------------- 13. Variar as cargas ----------------
        geral = MathTex(r"q_0 \cdot 2^{-n}", color=COR_TEXTO, font_size=52).move_to([0.5, -2.6, 0])
        self.fixar(geral)
        with self.fala("Assim ele pôde variar as cargas de forma controlada.") as t:
            self.play(FadeIn(geral, scale=1.4), Indicate(rot_barras, color=WHITE), run_time=t.duration * 0.5)
            self.play(geral.animate.scale(0.8).to_corner(UR), run_time=t.duration * 0.5)
        self.wait(0.5)

## Cena 4 — As três propriedades da força elétrica

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombPropriedades
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombPropriedades(CenaCoulomb):
    def construct(self):
        self.preparar()
        rng = np.random.default_rng(11)

        # Estado do par de cargas
        r = ValueTracker(1.0)        # distância (unidades físicas)
        q1 = ValueTracker(1.0)
        q2 = ValueTracker(1.0)
        theta = ValueTracker(0.0)    # orientação da reta entre as cargas
        centro = VectorizedPoint([-3.6, 1.0, 0])
        ESC = 1.2                    # unidades de tela por unidade de distância
        K_SETA = 1.4                 # comprimento da seta para F = 1

        def u():
            a = theta.get_value()
            return np.array([np.cos(a), np.sin(a), 0])

        def p1():
            return centro.get_center() - u() * r.get_value() * ESC / 2

        def p2():
            return centro.get_center() + u() * r.get_value() * ESC / 2

        def forca():
            return q1.get_value() * q2.get_value() / r.get_value() ** 2

        c1 = always_redraw(lambda: carga_2d(+1, 0.25 * np.sqrt(q1.get_value())).move_to(p1()))
        c2 = always_redraw(lambda: carga_2d(+1, 0.25 * np.sqrt(q2.get_value())).move_to(p2()))
        f1 = always_redraw(lambda: seta_direcao(p1(), -u(), K_SETA * forca(), maximo=2.6))
        f2 = always_redraw(lambda: seta_direcao(p2(), u(), K_SETA * forca(), maximo=2.6))
        estilo = {"cor": COR_COTA, "w": 2}
        realce = ValueTracker(0)
        linha = always_redraw(lambda: DashedLine(p1() - u() * 2.5, p2() + u() * 2.5,
                                                 color=estilo["cor"], stroke_width=estilo["w"] + realce.get_value(),
                                                 dash_length=0.12))
        linha.set_opacity(0)

        # ---------------- 1. Inverso do quadrado ----------------
        titulo = Text("Propriedade 1: inverso do quadrado", font_size=30, color=COR_TEXTO).to_corner(UL)
        with self.fala("Primeira propriedade: a força cai com o quadrado da distância.") as t:
            self.play(FadeIn(c1), FadeIn(c2), GrowArrow(f1), GrowArrow(f2), Write(titulo),
                      run_time=1.2)
            self.play(r.animate.set_value(4), run_time=max(t.duration - 1.2, 1))

        # ---------------- 2. Dobro da distância ----------------
        fantasma = seta(ORIGIN, RIGHT * K_SETA, opacidade=0.3)
        rot_f = MathTex("F", color=COR_TEXTO, font_size=36)
        rot_f4 = MathTex(r"F/4", color=COR_TEXTO, font_size=36)
        with self.fala("Dobrando a distância, a força cai para um quarto.") as t:
            self.play(r.animate.set_value(1), run_time=t.duration * 0.3)
            fantasma.next_to(f2, DOWN, buff=0.35).align_to(f2, LEFT)
            rot_f.next_to(fantasma, RIGHT, buff=0.15)
            self.play(FadeIn(fantasma), FadeIn(rot_f), run_time=t.duration * 0.15)
            self.play(r.animate.set_value(2), fantasma.animate.shift(RIGHT * ESC),
                      rot_f.animate.shift(RIGHT * ESC), run_time=t.duration * 0.35)
            rot_f4.next_to(f2, UP, buff=0.2)
            self.play(Flash(p2(), color=COR_TEXTO), FadeIn(rot_f4, shift=UP), run_time=t.duration * 0.2)

        # ---------------- 3. Triplo ----------------
        rot_f9 = MathTex(r"F/9", color=COR_TEXTO, font_size=36)
        with self.fala("Triplicando, cai para um nono.") as t:
            self.play(r.animate.set_value(3), fantasma.animate.shift(RIGHT * ESC / 2),
                      rot_f.animate.shift(RIGHT * ESC / 2), rot_f4.animate.shift(RIGHT * ESC / 2),
                      run_time=t.duration * 0.6)
            rot_f9.next_to(rot_f4, RIGHT, buff=0.3)
            self.play(ReplacementTransform(rot_f4, rot_f9), run_time=t.duration * 0.4)

        # ---------------- 4. Gráfico ----------------
        eixos = Axes(x_range=[0, 4.5, 1], y_range=[0, 3, 1], x_length=5, y_length=3.6,
                     axis_config={"color": COR_COTA, "include_tip": True, "tip_length": 0.18},
                     ).to_edge(RIGHT, buff=0.6).shift(DOWN * 0.4)
        lx = eixos.get_x_axis_label(MathTex("r", color=COR_TEXTO))
        ly = eixos.get_y_axis_label(MathTex("F", color=COR_TEXTO))
        ponto = always_redraw(lambda: Dot(eixos.c2p(r.get_value(), min(forca(), 3)), color=COR_FORCA))
        rastro = TracedPath(ponto.get_center, stroke_color=COR_FORCA, stroke_width=4)
        with self.fala("No gráfico, a curva despenca.") as t:
            self.play(FadeOut(VGroup(fantasma, rot_f, rot_f9)), Create(eixos), Write(lx), Write(ly),
                      run_time=1.2)
            self.add(ponto, rastro)
            self.play(r.animate.set_value(0.6), run_time=max((t.duration - 1.2) * 0.4, 0.8))
            self.play(r.animate.set_value(4), run_time=max((t.duration - 1.2) * 0.6, 1.2))

        # ---------------- 5. Força central ----------------
        titulo2 = Text("Propriedade 2: força central", font_size=30, color=COR_TEXTO).to_corner(UL)
        with self.fala("Segunda: a força aponta ao longo da linha entre as cargas.") as t:
            rastro.clear_updaters()
            self.play(FadeOut(VGroup(eixos, lx, ly, ponto, rastro)),
                      ReplacementTransform(titulo, titulo2),
                      centro.animate.move_to([0, -0.3, 0]), r.animate.set_value(2.5),
                      run_time=1.2)
            linha.set_opacity(1)
            self.add(linha)
            self.play(theta.animate.set_value(TAU), run_time=max(t.duration - 1.2, 1.5))
        theta.add_updater(lambda m, dt: m.increment_value(0.5 * dt))

        # ---------------- 6. Nenhuma direção especial ----------------
        tortas = VGroup()
        for _ in range(6):
            a = rng.uniform(0, TAU)
            base = rng.uniform(-3, 3, 3) * [1.6, 0.8, 0]
            tortas.add(seta_direcao(base, [np.cos(a), np.sin(a), 0], 1.0, cor=COR_ERRO, opacidade=0.6))
        cruzes = VGroup(*[Cross(s, stroke_color=RED, stroke_width=4).scale(0.6) for s in tortas])
        with self.fala("No espaço vazio, nenhuma direção é especial.") as t:
            self.play(LaggedStart(*[GrowArrow(s) for s in tortas], lag_ratio=0.1), run_time=t.duration * 0.45)
            self.play(LaggedStart(*[Create(c) for c in cruzes], lag_ratio=0.1), run_time=t.duration * 0.3)
            self.play(FadeOut(tortas), FadeOut(cruzes), run_time=t.duration * 0.25)

        # ---------------- 7. A reta entre as cargas ----------------
        with self.fala("A única direção marcada pelas cargas é a reta que as une.") as t:
            estilo.update(cor=COR_TEXTO, w=4)
            self.play(realce.animate(rate_func=there_and_back).set_value(6),
                      run_time=max(t.duration, 1))

        # ---------------- 8. Proporcional às cargas ----------------
        titulo3 = Text("Propriedade 3: produto das cargas", font_size=30, color=COR_TEXTO).to_corner(UL)
        mostrador = VGroup(
            MathTex("q_1 =", font_size=34), DecimalNumber(1, num_decimal_places=1, font_size=34),
            MathTex(r"\quad F =", font_size=34), DecimalNumber(1, num_decimal_places=2, font_size=34),
        ).arrange(RIGHT).set_color(COR_TEXTO).to_edge(DOWN, buff=0.5)
        mostrador[1].add_updater(lambda m: m.set_value(q1.get_value()))
        mostrador[3].add_updater(lambda m: m.set_value(forca() * r.get_value() ** 2))
        with self.fala("Terceira: a força é proporcional a cada carga.") as t:
            theta.clear_updaters()
            self.play(ReplacementTransform(titulo2, titulo3), theta.animate.set_value(0),
                      r.animate.set_value(3), FadeIn(mostrador), run_time=t.duration * 0.4)
            self.play(q1.animate.set_value(2), run_time=t.duration * 0.6)

        # ---------------- 9. Dobra e quadruplica ----------------
        rot_2f = MathTex("2F", color=COR_TEXTO, font_size=40).next_to(centro, UP, buff=1.0)
        rot_4f = MathTex("4F", color=COR_TEXTO, font_size=40).move_to(rot_2f)
        with self.fala("Dobre uma carga, e a força dobra. Dobre as duas, e ela quadruplica.") as t:
            self.play(FadeIn(rot_2f, scale=1.5), run_time=t.duration * 0.3)
            self.play(q2.animate.set_value(2), run_time=t.duration * 0.45)
            self.play(ReplacementTransform(rot_2f, rot_4f), run_time=t.duration * 0.25)

        # ---------------- 10. Produto q1 q2 ----------------
        pecas = VGroup(MathTex(r"\frac{1}{r^2}", font_size=48),
                       Text("reta entre as cargas", font_size=28),
                       MathTex(r"q_1\,q_2", font_size=48)).set_color(COR_TEXTO)
        pecas[0].move_to([-5, 2.2, 0]); pecas[1].move_to([0, 3.0, 0]); pecas[2].move_to([5, 2.2, 0])
        with self.fala("Ou seja, a força é proporcional ao produto q um vezes q dois.") as t:
            self.play(LaggedStart(*[FadeIn(p, shift=DOWN) for p in pecas], lag_ratio=0.3),
                      FadeOut(rot_4f), run_time=t.duration * 0.5)
            self.play(pecas.animate.arrange(RIGHT, buff=0.4).move_to([-3.9, 2.4, 0]),
                      run_time=t.duration * 0.5)

        # ---------------- 11. Proporcional ≠ crescente ----------------
        caixa = caixa_preta(UR, 6.9, 3.4)
        mini1 = Axes(x_range=[-1.5, 1.5], y_range=[-3, 3], x_length=2.4, y_length=2.0,
                     axis_config={"color": COR_COTA, "include_tip": False})
        mini2 = Axes(x_range=[-1.5, 1.5], y_range=[-7.5, 7.5], x_length=2.4, y_length=2.0,
                     axis_config={"color": COR_COTA, "include_tip": False})
        g1 = mini1.plot(lambda x: x ** 3, x_range=[-1.4, 1.4], color=COR_RESULTADO)
        g2 = mini2.plot(lambda x: -5 * x, x_range=[-1.4, 1.4], color=COR_RESULTADO)
        bloco1 = VGroup(mini1, g1, MathTex(r"y = x^3", font_size=30))
        bloco2 = VGroup(mini2, g2, MathTex(r"y = -5x", font_size=30))
        bloco1[2].next_to(mini1, UP, buff=0.1); bloco2[2].next_to(mini2, UP, buff=0.1)
        marca1 = MathTex(r"\times", color=RED, font_size=60)
        marca2 = MathTex(r"\checkmark", color=GREEN, font_size=60)
        legenda1 = Text("cresce, mas não é proporcional", font_size=15)
        legenda2 = Text("é proporcional, mas decresce", font_size=15)
        VGroup(bloco1, bloco2).arrange(RIGHT, buff=0.9).move_to(caixa).shift(UP * 0.2)
        legenda1.next_to(bloco1, DOWN, buff=0.1); legenda2.next_to(bloco2, DOWN, buff=0.1)
        marca1.next_to(bloco1[2], RIGHT); marca2.next_to(bloco2[2], RIGHT)
        aviso = Text("Proporcional ≠ crescente", font_size=30, color=COR_TEXTO).next_to(caixa, DOWN, buff=0.2)
        with self.fala("Atenção: proporcional não é o mesmo que crescente.") as t:
            self.play(FadeOut(VGroup(mostrador, linha)), centro.animate.move_to([-3.5, -1.5, 0]),
                      FadeIn(caixa), Write(aviso), run_time=t.duration * 0.3)
            self.play(Create(bloco1), Create(bloco2), run_time=t.duration * 0.4)
            self.play(FadeIn(marca1, scale=2), FadeIn(marca2, scale=2), FadeIn(legenda1), FadeIn(legenda2),
                      run_time=t.duration * 0.3)
        self.play(theta.animate.set_value(PI), run_time=1.5)

## Cena 5 — A fórmula, o sinal das cargas e as constantes

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombFormula
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombFormula(CenaCoulomb):
    def construct(self):
        self.preparar()

        # ---------------- 1. Montagem da equação ----------------
        p_fk = MathTex(r"\vec{F} = k\,", font_size=60)
        p_q = MathTex(r"q_1 q_2", font_size=60, color=COR_TEXTO)
        p_r = MathTex(r"\cdot\frac{1}{r^2}", font_size=60, color=COR_TEXTO)
        p_u = MathTex(r"\,\hat{r}", font_size=60, color=COR_TEXTO)
        montada = VGroup(p_fk, p_q, p_r, p_u).arrange(RIGHT, buff=0.45).move_to([0, 2.4, 0])
        alvos = [m.get_center() for m in montada]
        p_q.move_to([5, 2.6, 0]); p_r.move_to([-5, 2.6, 0]); p_u.move_to([0, 3.6, 0])
        chaves = VGroup()
        with self.fala("Juntando as três propriedades, temos a lei de Coulomb.") as t:
            self.play(FadeIn(VGroup(p_q, p_r, p_u)), run_time=0.5)
            self.play(Write(p_fk), *[m.animate.move_to(a) for m, a in zip((p_q, p_r, p_u), alvos[1:])],
                      run_time=t.duration * 0.4)
            for peca, txt in ((p_q, "Prop. 1.3.3"), (p_r, "Prop. 1.3.1"), (p_u, "Prop. 1.3.2")):
                b = Brace(peca, DOWN, buff=0.1, color=COR_COTA)
                b.shift(UP * (p_r.get_bottom()[1] - 0.15 - b.get_top()[1]))
                chaves.add(VGroup(b, Text(txt, font_size=20, color=COR_COTA).next_to(b, DOWN, buff=0.1)))
            self.play(LaggedStart(*[GrowFromCenter(c) for c in chaves], lag_ratio=0.3),
                      run_time=max(t.duration * 0.6 - 0.5, 1))

        compacta = MathTex(r"\vec{F} = k\,\frac{q_1 q_2}{r^2}\,\hat{r}", font_size=60).move_to([0, 2.4, 0])

        # Par de cargas
        q2 = ValueTracker(1.0)
        theta = ValueTracker(0.0)
        sep = ValueTracker(3.0)
        centro = np.array([0, -1.4, 0])

        def u():
            a = theta.get_value()
            return np.array([np.cos(a), np.sin(a), 0])

        def p1():
            return centro - u() * sep.get_value() / 2

        def p2():
            return centro + u() * sep.get_value() / 2

        def carga_variavel():
            v = q2.get_value()
            cor = interpolate_color(COR_CARGA_NEG, COR_CARGA, (v + 1) / 2)
            c = carga_2d(1 if v >= 0 else -1, 0.3, cor=cor)
            return c.move_to(p2())

        c1 = always_redraw(lambda: carga_2d(+1, 0.3).move_to(p1()))
        c2 = always_redraw(carga_variavel)
        rot1 = always_redraw(lambda: MathTex("q_1", font_size=32, color=COR_TEXTO).next_to(p1(), UP, buff=0.4))
        rot2 = always_redraw(lambda: MathTex("q_2", font_size=32, color=COR_TEXTO).next_to(p2(), UP, buff=0.4))
        mostrar_r = ValueTracker(0)
        r_hat = always_redraw(lambda: seta_direcao(p1() - u() * 0.32, -u(), 0.9 * mostrar_r.get_value(),
                                                   cor=COR_UNITARIO, espessura=4))
        rot_rhat = always_redraw(lambda: MathTex(r"\hat{r}", font_size=34).set_opacity(mostrar_r.get_value())
                                 .move_to(p1() - u() * 0.8 + rotacao_z(u(), PI / 2) * 0.35))

        # ---------------- 2. r chapéu ----------------
        with self.fala("O vetor r chapéu tem módulo um.") as t:
            self.play(FadeOut(chaves), ReplacementTransform(VGroup(p_fk, p_q, p_r, p_u), compacta),
                      FadeIn(c1), FadeIn(c2), FadeIn(rot1), FadeIn(rot2), run_time=t.duration * 0.5)
            self.add(r_hat, rot_rhat)
            self.play(mostrar_r.animate.set_value(1), run_time=t.duration * 0.25)
            self.play(mostrar_r.animate(rate_func=there_and_back).set_value(1.4), run_time=t.duration * 0.25)

        # ---------------- 3. Placa de mão única ----------------
        def placa():
            d = -u()
            base = RoundedRectangle(width=1.1, height=0.42, corner_radius=0.06, color=WHITE,
                                    fill_opacity=1, stroke_width=0)
            ponta = Arrow(LEFT * 0.4, RIGHT * 0.4, buff=0, color=BLACK, stroke_width=6, tip_length=0.15)
            g = VGroup(base, ponta).scale(0.8)
            g.rotate(np.arctan2(d[1], d[0]))
            return g.move_to(p1() + d * 1.0 + rotacao_z(d, -PI / 2) * 0.75)

        sinal_transito = always_redraw(placa)
        with self.fala("Ele só indica direção, como a placa de mão única.") as t:
            self.play(FadeIn(sinal_transito), run_time=0.5)
            self.play(theta.animate.set_value(PI / 2), run_time=max(t.duration * 0.5 - 0.25, 0.8))
            self.play(theta.animate.set_value(0), run_time=max(t.duration * 0.5 - 0.25, 0.8))

        # ---------------- 4. Repulsão ----------------
        mostrar_f = ValueTracker(0)

        def f_em_1():
            return seta_direcao(p1(), -u(), 1.3 * q2.get_value() * mostrar_f.get_value(), espessura=6)

        def f_em_2():
            return seta_direcao(p2(), u(), 1.3 * q2.get_value() * mostrar_f.get_value(), espessura=6)

        f1 = always_redraw(f_em_1)
        f2 = always_redraw(f_em_2)
        with self.fala("Se q um vezes q dois é positivo, a força segue r chapéu: repulsão.") as t:
            self.add(f1, f2)
            self.play(FadeOut(sinal_transito), mostrar_f.animate.set_value(1), run_time=t.duration * 0.5)
            self.play(sep.animate(rate_func=there_and_back).set_value(3.8), run_time=t.duration * 0.5)

        # ---------------- 5. Troca de sinal ----------------
        leitura = VGroup(MathTex("q_2 =", font_size=34, color=COR_TEXTO),
                         DecimalNumber(1, num_decimal_places=2, include_sign=True, font_size=34,
                                       color=COR_TEXTO)).arrange(RIGHT).to_edge(DOWN, buff=0.4)
        leitura[1].add_updater(lambda m: m.set_value(q2.get_value()))
        with self.fala("Agora vamos trocar o sinal de q dois.") as t:
            self.play(FadeIn(leitura), run_time=0.4)
            self.play(q2.animate.set_value(-1), run_time=max(t.duration - 0.4, 1.5), rate_func=linear)

        # ---------------- 6. Atração ----------------
        with self.fala("Produto negativo: a força inverte. É atração.") as t:
            self.play(sep.animate(rate_func=there_and_back).set_value(2.2), run_time=t.duration * 0.6)
            self.play(Indicate(VGroup(f1, f2), color=COR_TEXTO), run_time=t.duration * 0.4)

        # ---------------- 7. Valor de k ----------------
        val_k = MathTex(r"k = 8{,}99\times10^{9}\ \mathrm{N\,m^2/C^2}", font_size=40, color=COR_TEXTO)
        val_k.move_to([0, 1.1, 0])
        with self.fala("A constante k vale cerca de 8,99 vezes dez à nona.") as t:
            self.play(Indicate(compacta[0][3], color=COR_TEXTO, scale_factor=1.6), run_time=t.duration * 0.4)
            self.play(FadeIn(val_k, shift=DOWN * 1.2), run_time=t.duration * 0.6)

        # ---------------- 8. 1/(4 pi eps0) ----------------
        val_k2 = MathTex(r"k = \frac{1}{4\pi\varepsilon_0} = 8{,}99\times10^{9}\ \mathrm{N\,m^2/C^2}",
                         font_size=40, color=COR_TEXTO).move_to(val_k)
        val_e0 = MathTex(r"\varepsilon_0 = 8{,}85\times10^{-12}\ \mathrm{C^2/N\,m^2}", font_size=36,
                         color=COR_TEXTO).next_to(val_k2, DOWN, buff=0.25)
        with self.fala("Também a escrevemos como um sobre quatro pi épsilon zero.") as t:
            self.play(TransformMatchingShapes(val_k, val_k2), sep.animate.set_value(3.6), run_time=t.duration * 0.55)
            self.play(FadeIn(val_e0, shift=UP), run_time=t.duration * 0.45)

        # ---------------- 9. O coulomb ----------------
        fio = Line([-5.5, -2.9, 0], [5.5, -2.9, 0], color=COR_COBRE, stroke_width=10)
        pontos = VGroup(*[Dot([x, -2.9, 0], radius=0.07, color=COR_CARGA_NEG) for x in np.linspace(-5.4, 5.4, 18)])

        def correr(m, dt):
            m.shift(RIGHT * 1.6 * dt)
            if m.get_center()[0] > 5.45:
                m.shift(LEFT * 10.9)

        for p in pontos:
            p.add_updater(correr)
        unidade = MathTex(r"1\ \mathrm{C} = (1\ \mathrm{A})(1\ \mathrm{s})", r"\qquad i = \frac{dq}{dt}",
                          font_size=40, color=COR_TEXTO).move_to([0, -2.0, 0])
        with self.fala("A unidade de carga é o coulomb: um ampère vezes um segundo.") as t:
            self.play(FadeOut(VGroup(c1, c2, rot1, rot2, r_hat, rot_rhat, f1, f2, leitura)),
                      Create(fio), FadeIn(pontos), run_time=t.duration * 0.45)
            self.play(Write(unidade), run_time=t.duration * 0.55)

        # ---------------- 10. Módulos e sentidos ----------------
        modulo = MathTex(r"F = \frac{1}{4\pi\varepsilon_0}\,\frac{|q_1|\,|q_2|}{r^2}", font_size=54)
        modulo.move_to([0, -0.6, 0])
        moldura = SurroundingRectangle(modulo, color=YELLOW, buff=0.25)
        with self.fala("Nos problemas, os módulos dão o tamanho. Os sinais dão o sentido.") as t:
            self.play(FadeOut(VGroup(val_k2, val_e0, unidade)), Write(modulo), run_time=t.duration * 0.5)
            self.play(Create(moldura), Indicate(compacta, color=COR_RESULTADO), run_time=t.duration * 0.5)
        self.wait(0.5)

## Cena 6 — Forma vetorial com vetores posição

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombVetorial
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombVetorial(CenaCoulomb):
    def construct(self):
        self.preparar()

        plano = NumberPlane(x_range=[-7, 7, 1], y_range=[-4, 4, 1],
                            background_line_style={"stroke_color": COR_COTA, "stroke_opacity": 0.2,
                                                   "stroke_width": 1},
                            axis_config={"stroke_color": COR_COTA, "stroke_opacity": 0.5})
        origem = Dot(ORIGIN, color=WHITE)
        rot_o = MathTex("O", font_size=34).next_to(origem, DL, buff=0.1)

        # Relógio próprio do movimento (pode desacelerar)
        vel = ValueTracker(1.0)
        fase = ValueTracker(0.0)
        fase.add_updater(lambda m, dt: m.increment_value(dt * vel.get_value()))
        self.add(fase)

        def pos1():
            s = fase.get_value()
            return np.array([-2.4 + 1.0 * np.cos(0.5 * s), 0.8 + 0.5 * np.sin(0.5 * s), 0])

        def pos2():
            s = fase.get_value()
            return np.array([2.2 + 1.0 * np.cos(0.7 * s + 1), -1.5 + 0.7 * np.sin(0.7 * s + 1), 0])

        c1 = always_redraw(lambda: carga_2d(+1).move_to(pos1()))
        c2 = always_redraw(lambda: carga_2d(+1).move_to(pos2()))
        r1 = always_redraw(lambda: seta(ORIGIN, pos1() - normalize(pos1()) * 0.25, cor=COR_UNITARIO, espessura=4))
        r2 = always_redraw(lambda: seta(ORIGIN, pos2() - normalize(pos2()) * 0.25, cor=COR_UNITARIO, espessura=4))
        rot_r1 = always_redraw(lambda: MathTex(r"\vec{r}_1", font_size=34, color=COR_TEXTO)
                               .move_to(pos1() * 0.5 + rotacao_z(normalize(pos1()), PI / 2) * 0.35))
        rot_r2 = always_redraw(lambda: MathTex(r"\vec{r}_2", font_size=34, color=COR_TEXTO)
                               .move_to(pos2() * 0.5 + rotacao_z(normalize(pos2()), PI / 2) * 0.35))

        # ---------------- 1. Plano ----------------
        with self.fala("Agora, sem depender do desenho: vetores posição.") as t:
            self.play(Create(plano, lag_ratio=0.05), run_time=t.duration * 0.6)
            self.play(FadeIn(origem, scale=3), Write(rot_o), run_time=t.duration * 0.4)

        # ---------------- 2. r1 ----------------
        with self.fala("r um vai da origem até a carga um.") as t:
            self.play(FadeIn(c1), GrowArrow(r1), Write(rot_r1), run_time=t.duration * 0.5)
            self.wait(t.duration * 0.5)

        # ---------------- 3. r2 ----------------
        with self.fala("r dois vai até a carga dois.") as t:
            self.play(FadeIn(c2), GrowArrow(r2), Write(rot_r2), run_time=t.duration * 0.5)
            self.wait(t.duration * 0.5)

        # ---------------- 4. Diferença ----------------
        def seta_dif():
            a, b = pos2(), pos1()
            d = b - a
            n = np.linalg.norm(d)
            linha = DashedLine(a + d / n * 0.28, b - d / n * 0.28, color=COR_COTA, stroke_width=4,
                               dash_length=0.12)
            linha.add_tip(tip_length=0.18)
            return linha

        with self.fala("A diferença r um menos r dois liga a carga dois à carga um.") as t:
            self.play(vel.animate.set_value(0), run_time=0.6)
            a, b = pos1(), pos2()
            menos_r2 = seta(ORIGIN, -b, cor=COR_UNITARIO, espessura=4)
            rot_m = MathTex(r"-\vec{r}_2", font_size=30, color=COR_TEXTO).next_to(menos_r2.get_end(), DOWN)
            self.play(GrowArrow(menos_r2), FadeIn(rot_m), run_time=t.duration * 0.25)
            self.play(menos_r2.animate.shift(a), rot_m.animate.shift(a), run_time=t.duration * 0.3)
            dif = seta_dif()
            self.play(ReplacementTransform(menos_r2, dif), FadeOut(rot_m), run_time=t.duration * 0.25)
            self.remove(dif)
            dif = always_redraw(seta_dif)
            self.add(dif)
            self.play(vel.animate.set_value(1), run_time=0.5)

        # ---------------- 5. Distância ----------------
        distancia = VGroup(MathTex(r"|\vec{r}_1-\vec{r}_2| =", font_size=32),
                           DecimalNumber(0, num_decimal_places=2, font_size=32)).arrange(RIGHT)
        distancia.set_color(COR_TEXTO)
        distancia.add_updater(lambda m: m.move_to((pos1() + pos2()) / 2 + DOWN * 0.1 + RIGHT * 1.6))
        distancia[1].add_updater(lambda m: m.set_value(np.linalg.norm(pos1() - pos2())))
        fundo_d = always_redraw(lambda: BackgroundRectangle(distancia, color=FUNDO, fill_opacity=0.8, buff=0.08))
        with self.fala("Seu módulo é a distância entre elas.") as t:
            self.play(FadeIn(fundo_d), FadeIn(distancia), run_time=t.duration * 0.5)
            self.wait(t.duration * 0.5)

        # ---------------- 6. r chapéu ----------------
        r_hat = always_redraw(lambda: seta_direcao(pos1() + normalize(pos1() - pos2()) * 0.27,
                                                   pos1() - pos2(), 1.0, cor=COR_UNITARIO, espessura=5))
        rot_rhat = always_redraw(lambda: MathTex(r"\hat{r}", font_size=32)
                                 .move_to(pos1() + normalize(pos1() - pos2()) * 1.5))
        with self.fala("Dividindo por ele, obtemos r chapéu.") as t:
            self.play(GrowArrow(r_hat), Write(rot_rhat), run_time=t.duration * 0.6)
            self.play(Indicate(rot_rhat), run_time=t.duration * 0.4)

        # ---------------- 7. Equação vetorial ----------------
        caixa = caixa_preta(UL, 7.0, 1.6)
        eq = MathTex(r"\vec{F}_{1\leftarrow 2} = k\,q_1 q_2\,",
                     r"\frac{\vec{r}_1-\vec{r}_2}{|\vec{r}_1-\vec{r}_2|^{3}}", font_size=46).move_to(caixa)
        mostrar = ValueTracker(0)
        forca = always_redraw(lambda: seta_direcao(
            pos1() + normalize(pos1() - pos2()) * 0.27, pos1() - pos2(),
            mostrar.get_value() * 30.0 / max(np.linalg.norm(pos1() - pos2()), 1.5) ** 2,
            maximo=2.2, espessura=7))
        with self.fala("A força fica: k, q um, q dois, vezes r um menos r dois, sobre o módulo ao cubo.") as t:
            self.play(FadeIn(caixa), Write(eq), run_time=t.duration * 0.6)
            self.add(forca)
            self.play(mostrar.animate.set_value(1), run_time=t.duration * 0.4)

        # ---------------- 8. Cuidado com o cubo ----------------
        aviso = MathTex(r"\frac{r}{r^{3}}", "=", r"\frac{1}{r^{2}}", font_size=46, color=COR_TEXTO)
        aviso.next_to(caixa, DOWN, buff=0.3).align_to(caixa, LEFT).shift(RIGHT * 0.4)
        fundo_a = BackgroundRectangle(aviso, color=FUNDO, fill_opacity=0.85, buff=0.12)
        with self.fala("Cuidado: esse cubo não significa queda com o cubo.") as t:
            self.play(Indicate(eq[1], color=COR_ERRO, scale_factor=1.15), run_time=t.duration * 0.35)
            self.play(FadeIn(fundo_a), Write(aviso), run_time=t.duration * 0.35)
            self.play(Create(Cross(aviso[0], stroke_color=RED, stroke_width=4)), Indicate(aviso[2]),
                      run_time=t.duration * 0.3)

        # ---------------- 9. Cargas em repouso ----------------
        repouso = Text("cargas em repouso", font_size=34, color=COR_TEXTO).to_corner(DR, buff=0.7)
        with self.fala("Lembre: a lei vale para cargas em repouso.") as t:
            self.play(vel.animate.set_value(0), Write(repouso), run_time=t.duration, rate_func=smooth)
        self.play(Indicate(repouso), run_time=1)

## Cena 7 — Gravidade versus eletricidade no núcleo de ferro

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombNucleo
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombNucleo(CenaCoulomb):
    def construct(self):
        self.preparar()
        rng = np.random.default_rng(5)

        partes_c = [r"\vec{F} = ", r"k", r"\,q_1 q_2", r"\,\frac{1}{r^2}", r"\,\hat{r}"]
        partes_n = [r"\vec{F} = ", r"G", r"\,m_1 m_2", r"\,\frac{1}{r^2}", r"\,\hat{r}"]
        coulomb = MathTex(*partes_c, font_size=56)
        coulomb[1].set_color(COR_TEXTO)
        newton = MathTex(*partes_n, font_size=56)
        newton[1].set_color(COR_TEXTO)
        self.fixar(coulomb, newton)

        # ---------------- 1. Coulomb vira Newton ----------------
        with self.fala("A lei de Coulomb lembra a gravitação de Newton.") as t:
            self.play(Write(coulomb), run_time=t.duration * 0.4)
            copia = coulomb.copy()
            self.fixar(copia)
            self.add(copia)
            self.play(TransformMatchingTex(copia, newton), coulomb.animate.shift(LEFT * 3.4 + UP * 2.4),
                      run_time=t.duration * 0.6)
            self.remove(copia)

        # ---------------- 2. Inverso do quadrado ----------------
        with self.fala("As duas caem com o inverso do quadrado.") as t:
            self.play(newton.animate.shift(RIGHT * 3.4 + UP * 2.4), run_time=t.duration * 0.4)
            self.play(Indicate(coulomb[3], color=COR_TEXTO, scale_factor=1.4),
                      Indicate(newton[3], color=COR_TEXTO, scale_factor=1.4), run_time=t.duration * 0.6)

        # ---------------- 3. Só atrai x atrai ou repele ----------------
        relogio = self.relogio
        m1 = always_redraw(lambda: Dot([-3.4 - 1.2 - 0.5 * np.sin(1.5 * relogio.get_value()), -0.3, 0],
                                       radius=0.3, color=GRAY))
        m2 = always_redraw(lambda: Dot([-3.4 + 1.2 + 0.5 * np.sin(1.5 * relogio.get_value()), -0.3, 0],
                                       radius=0.3, color=GRAY))
        fg1 = always_redraw(lambda: seta_direcao(m1.get_center(), RIGHT, 0.9))
        fg2 = always_redraw(lambda: seta_direcao(m2.get_center(), LEFT, 0.9))

        def sinal_q2():
            return 1 if np.sin(0.9 * relogio.get_value()) >= 0 else -1

        e1 = always_redraw(lambda: carga_2d(+1, 0.3).move_to([3.4 - 1.5, -0.3, 0]))
        e2 = always_redraw(lambda: carga_2d(sinal_q2(), 0.3).move_to([3.4 + 1.5, -0.3, 0]))
        fe1 = always_redraw(lambda: seta_direcao([3.4 - 1.5, -0.3, 0], LEFT * sinal_q2(), 0.9))
        fe2 = always_redraw(lambda: seta_direcao([3.4 + 1.5, -0.3, 0], RIGHT * sinal_q2(), 0.9))
        demo = VGroup(m1, m2, fg1, fg2, e1, e2, fe1, fe2)
        rot_g = Text("só atrai", font_size=28, color=COR_TEXTO).move_to([-3.4, -1.6, 0])
        rot_e = Text("atrai ou repele", font_size=28, color=COR_TEXTO).move_to([3.4, -1.6, 0])
        with self.fala("Mas a gravidade só atrai. A força elétrica atrai ou repele.") as t:
            self.play(FadeIn(demo), run_time=0.8)
            self.play(Write(rot_g), Write(rot_e), run_time=max(t.duration - 0.8, 1))

        # ---------------- 4. Núcleo de ferro (3D) ----------------
        nucleo = VGroup()
        tipos = [1] * 26 + [0] * 30
        rng.shuffle(tipos)
        pontos = (pontos_fibonacci(6, 0.33) + pontos_fibonacci(18, 0.72) + pontos_fibonacci(32, 1.1))
        for p, tp in zip(pontos, tipos):
            esf = carga_3d(1 if tp else 0, 0.24, resolucao=(5, 6),
                           cor=COR_CARGA if tp else COR_NEUTRA)
            esf.move_to(p)
            esf.proton = bool(tp)
            nucleo.add(esf)
        nucleo.add_updater(lambda m, dt: m.rotate(0.35 * dt, axis=OUT, about_point=m.get_center()))
        cota = Line(ORIGIN, RIGHT * 1.35, color=COR_COTA, stroke_width=4).shift(OUT * 1.6)
        rot_raio = MathTex(r"R = 4{,}0\times10^{-15}\ \mathrm{m}", font_size=36, color=COR_TEXTO).to_corner(DL)
        titulo_fe = Text("núcleo de ferro: 26 prótons", font_size=30, color=COR_TEXTO).to_corner(UL)
        self.fixar(rot_raio, titulo_fe)
        with self.fala("Vamos ao núcleo de um átomo de ferro.") as t:
            self.play(FadeOut(VGroup(demo, rot_g, rot_e, coulomb, newton)), run_time=0.8)
            self.move_camera(phi=60 * DEGREES, theta=-45 * DEGREES, zoom=1.2, run_time=1,
                             added_anims=[FadeIn(nucleo, lag_ratio=0.02)])
            self.begin_ambient_camera_rotation(rate=0.05)
            self.play(Create(cota), FadeIn(rot_raio), FadeIn(titulo_fe), run_time=max(t.duration - 1.8, 0.8))

        # ---------------- 5. Dois prótons ----------------
        protons = [m for m in nucleo if m.proton][:2]
        destino = [np.array([1.2, -1.2, 0]), np.array([3.2, -1.2, 0])]
        with self.fala("Pegue dois prótons a quatro fentômetros.") as t:
            nucleo.clear_updaters()
            offsets = [p.get_center() - nucleo.get_center() for p in protons]
            for p in protons:
                nucleo.remove(p)
                self.add(p)
            self.play(nucleo.animate.shift(LEFT * 3.0 + UP * 1.6), FadeOut(cota),
                      *[p.animate.move_to(d) for p, d in zip(protons, destino)],
                      run_time=t.duration * 0.7)
            nucleo.add_updater(lambda m, dt: m.rotate(0.35 * dt, axis=OUT, about_point=m.get_center()))
            self.play(*[Indicate(p, color=COR_TEXTO) for p in protons], run_time=t.duration * 0.3)

        # ---------------- 6. 14 N ----------------
        mostrar = ValueTracker(0)
        fp1 = always_redraw(lambda: seta_direcao(destino[0] + LEFT * 0.25, LEFT, 2.0 * mostrar.get_value(),
                                                 espessura=8, ponta=0.25))
        fp2 = always_redraw(lambda: seta_direcao(destino[1] + RIGHT * 0.25, RIGHT, 2.0 * mostrar.get_value(),
                                                 espessura=8, ponta=0.25))
        leitura = VGroup(MathTex(r"F_{el} \approx", font_size=40),
                         DecimalNumber(0, num_decimal_places=1, unit=r"\ \mathrm{N}", font_size=40))
        leitura.arrange(RIGHT).set_color(COR_TEXTO).to_corner(UR)
        leitura[1].add_updater(lambda m: m.set_value(14.4 * mostrar.get_value()))
        self.fixar(leitura)
        with self.fala("Eles se repelem com cerca de 14 newtons.") as t:
            self.add(fp1, fp2, leitura)
            self.play(mostrar.animate.set_value(1), run_time=t.duration * 0.8)

        # ---------------- 7. Gravidade e escala log ----------------
        regua = VGroup()
        topo, base = np.array([6.3, 1.4, 0]), np.array([6.3, -3.2, 0])
        regua.add(Line(topo, base, color=COR_COTA, stroke_width=3))
        for k in range(37):
            y = interpolate(topo, base, k / 36)
            regua.add(Line(y + LEFT * (0.18 if k % 5 == 0 else 0.09), y, color=COR_COTA, stroke_width=2))
        r_top = MathTex(r"10^{1}\,\mathrm{N}", font_size=26, color=COR_TEXTO).next_to(topo, LEFT, buff=0.3)
        r_base = MathTex(r"10^{-35}\,\mathrm{N}", font_size=26, color=COR_TEXTO).next_to(base, LEFT, buff=0.3)
        cursor = Dot(topo, color=COR_FORCA, radius=0.1)
        grav = MathTex(r"F_{grav} \approx 1{,}2\times10^{-35}\ \mathrm{N}", font_size=36, color=COR_GAUSS)
        grav.next_to(leitura, DOWN, aligned_edge=RIGHT)
        fg_p = VGroup(seta(destino[0] + RIGHT * 0.25, destino[0] + RIGHT * 0.32, cor=GRAY, espessura=2),
                      seta(destino[1] + LEFT * 0.25, destino[1] + LEFT * 0.32, cor=GRAY, espessura=2))
        self.fixar(regua, r_top, r_base, cursor, grav)
        with self.fala("A gravidade entre eles: 1,2 vezes dez a menos trinta e cinco newtons.") as t:
            self.play(FadeIn(regua), FadeIn(r_top), FadeIn(r_base), FadeIn(cursor), Write(grav),
                      FadeIn(fg_p), run_time=1.2)
            self.play(cursor.animate.move_to(base), run_time=max(t.duration - 1.2, 2), rate_func=linear)

        # ---------------- 8. Interação forte ----------------
        forte = Text("interação forte", font_size=36, color=COR_TEXTO).to_edge(DOWN)
        self.fixar(forte)
        with self.fala("A gravidade não segura o núcleo. Quem segura é a interação forte.") as t:
            nucleo.clear_updaters()
            self.play(FadeOut(VGroup(fp1, fp2, fg_p, regua, r_top, r_base, cursor, grav, leitura)),
                      nucleo.animate.shift(RIGHT * 3.0 + DOWN * 1.6),
                      *[p.animate.move_to(nucleo.get_center() + RIGHT * 3.0 + DOWN * 1.6 + o) for p, o in zip(protons, offsets)],
                      run_time=t.duration * 0.5)
            self.play(Write(forte),
                      LaggedStart(*[Indicate(m, color=WHITE, scale_factor=1.3) for m in list(nucleo)[::4]],
                                  lag_ratio=0.1), run_time=t.duration * 0.5)
        self.wait(1)

## Cena 8 — Superposição: Exemplo 21.01 (Halliday)

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombSuperposicao
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombSuperposicao(CenaCoulomb):
    def construct(self):
        self.preparar()

        Q1 = np.array([-2.5, -1.5, 0])
        R = 4.0                       # R = 0,0200 m na tela
        Q2 = Q1 + RIGHT * R
        ESC = 1.2                     # unidades de tela por 10^-24 N
        F12, F13 = 1.15, 2.05         # em 10^-24 N

        # ---------------- 1. Equação da superposição ----------------
        eq = MathTex(r"\vec{F}_{1,tot}", "=", r"\vec{F}_{12}", "+", r"\vec{F}_{13}", "+",
                     r"\vec{F}_{14}", "+", r"\cdots", "+", r"\vec{F}_{1n}", font_size=46).to_edge(UP, buff=0.3)
        with self.fala("Com várias cargas, as forças se somam como vetores.") as t:
            self.play(Write(eq), run_time=t.duration * 0.5)
            self.play(LaggedStart(*[Indicate(eq[i], color=COR_TEXTO) for i in (2, 4, 6, 10)], lag_ratio=0.25),
                      run_time=t.duration * 0.5)

        # ---------------- 2. Primeiro arranjo ----------------
        eixo = NumberLine(x_range=[-7, 7, 1], length=14, include_ticks=False, color=COR_COTA).move_to([0, Q1[1], 0])
        rot_x = MathTex("x", font_size=32, color=COR_COTA).next_to(eixo.get_right(), UP, buff=0.1)
        c1 = carga_2d(+1).move_to(Q1)
        c2 = carga_2d(+1).move_to(Q2)
        r1 = MathTex("q_1", font_size=32, color=COR_TEXTO).next_to(c1, DOWN, buff=0.25)
        r2 = MathTex("q_2", font_size=32, color=COR_TEXTO).next_to(c2, DOWN, buff=0.25)
        cota = DoubleArrow(Q1 + DOWN * 1.0, Q2 + DOWN * 1.0, buff=0, color=COR_COTA, stroke_width=2,
                           tip_length=0.15)
        rot_R = MathTex("R", font_size=32, color=COR_COTA).next_to(cota, DOWN, buff=0.1)
        with self.fala("Primeiro arranjo: duas cargas positivas no eixo x.") as t:
            self.play(GrowFromCenter(eixo), FadeIn(rot_x), run_time=t.duration * 0.35)
            self.play(GrowFromCenter(c1), GrowFromCenter(c2), Write(r1), Write(r2), run_time=t.duration * 0.35)
            self.play(GrowFromCenter(cota), Write(rot_R), run_time=t.duration * 0.3)

        # ---------------- 3. F12 ----------------
        m12 = ValueTracker(0)
        f12 = always_redraw(lambda: seta_direcao(Q1, LEFT, ESC * F12 * m12.get_value()))
        rf12 = always_redraw(lambda: MathTex(r"\vec{F}_{12}", font_size=30, color=COR_TEXTO)
                             .next_to(Q1 + LEFT * ESC * F12 * m12.get_value() / 2, UP, buff=0.3))
        valor = VGroup(MathTex(r"F_{12} =", font_size=34), DecimalNumber(0, num_decimal_places=2, font_size=34),
                       MathTex(r"\times10^{-24}\ \mathrm{N},\ 180^\circ", font_size=34))
        valor.arrange(RIGHT, buff=0.12).set_color(COR_TEXTO).to_corner(DL, buff=0.4)
        valor[1].add_updater(lambda m: m.set_value(F12 * m12.get_value()))
        with self.fala("A partícula 1 é repelida com 1,15 vezes dez a menos vinte e quatro newtons.") as t:
            self.add(f12, rf12)
            self.play(m12.animate.set_value(1), FadeIn(valor), run_time=t.duration * 0.8)

        # ---------------- 4. Partícula 3 ----------------
        Q3 = Q1 + RIGHT * 0.75 * R
        c3 = carga_2d(-1).move_to(Q3 + UP * 3.5)
        r3 = MathTex("q_3", font_size=32, color=COR_TEXTO)
        r3.add_updater(lambda m: m.next_to(c3, UP, buff=0.2))
        with self.fala("Agora, uma carga negativa entre elas.") as t:
            self.play(FadeIn(c3), FadeIn(r3), run_time=0.4)
            self.play(MoveAlongPath(c3, ArcBetweenPoints(Q3 + UP * 3.5, Q3, angle=-PI / 3)),
                      run_time=max(t.duration - 0.4, 1))

        # ---------------- 5. F13 ----------------
        m13 = ValueTracker(0)
        f13 = always_redraw(lambda: seta_direcao(Q1, RIGHT, ESC * F13 * m13.get_value()))
        rf13 = always_redraw(lambda: MathTex(r"\vec{F}_{13}", font_size=30, color=COR_TEXTO)
                             .next_to(Q1 + RIGHT * ESC * F13 * m13.get_value() / 2, UP, buff=0.3))
        valor13 = MathTex(r"F_{13} = 2{,}05\times10^{-24}\ \mathrm{N},\ 0^\circ", font_size=34,
                          color=COR_TEXTO).next_to(valor, UP, aligned_edge=LEFT)
        with self.fala("Ela puxa a partícula 1 para a direita com 2,05 vezes dez a menos vinte e quatro.") as t:
            self.add(f13, rf13)
            self.play(m13.animate.set_value(1), FadeIn(valor13), run_time=t.duration * 0.8)

        # ---------------- 6. Soma colinear ----------------
        resultado = MathTex(r"F_{1,tot} = 9{,}0\times10^{-25}\ \mathrm{N},\ 0^\circ", font_size=34,
                            color=COR_RESULTADO).next_to(valor13, UP, aligned_edge=LEFT)
        with self.fala("Mesma direção, sentidos opostos: basta subtrair.") as t:
            copia = seta_direcao(Q1, LEFT, ESC * F12, opacidade=0.8)
            self.add(copia)
            self.play(copia.animate.shift(RIGHT * ESC * F13 + UP * 0.25), run_time=t.duration * 0.4)
            res = seta(Q1 + UP * 0.25, Q1 + UP * 0.25 + RIGHT * ESC * (F13 - F12), cor=COR_RESULTADO, espessura=7)
            self.play(GrowArrow(res), Write(resultado), run_time=t.duration * 0.6)

        # ---------------- 7. Terceiro arranjo: q4 a 60° ----------------
        ang = ValueTracker(0.0)
        dist = 0.75 * R

        def p4():
            a = ang.get_value()
            return Q1 + dist * np.array([np.cos(a), np.sin(a), 0])

        c4 = always_redraw(lambda: carga_2d(-1).move_to(p4()))
        r4 = always_redraw(lambda: MathTex("q_4", font_size=32, color=COR_TEXTO).next_to(p4(), UR, buff=0.1))
        arco = always_redraw(lambda: Arc(radius=0.8, start_angle=0, angle=max(ang.get_value(), 0.001),
                                         arc_center=Q1, color=COR_TEXTO, stroke_width=3))
        leit_ang = always_redraw(lambda: DecimalNumber(ang.get_value() / DEGREES, num_decimal_places=0,
                                                       unit=r"^\circ", font_size=30, color=COR_TEXTO)
                                 .move_to(Q1 + 1.25 * np.array([np.cos(ang.get_value() / 2),
                                                                np.sin(ang.get_value() / 2), 0])))
        f14 = always_redraw(lambda: seta_direcao(Q1, p4() - Q1, ESC * F13))
        rf14 = always_redraw(lambda: MathTex(r"\vec{F}_{14}", font_size=30, color=COR_TEXTO)
                             .move_to(Q1 + (p4() - Q1) * 0.55 + rotacao_z(normalize(p4() - Q1), -PI / 2) * 0.4))
        with self.fala("Terceiro arranjo: a carga negativa sai do eixo.") as t:
            self.play(FadeOut(VGroup(copia, res, resultado, valor13, f13, rf13)), run_time=0.6)
            self.remove(c3, r3)
            self.add(c4, r4, arco, leit_ang, f14, rf14)
            self.play(ang.animate.set_value(60 * DEGREES), run_time=max(t.duration - 0.6, 1.2))

        # ---------------- 8. Não colineares ----------------
        ext12 = DashedLine(Q1 + RIGHT * 6, Q1 + LEFT * 3.5, color=COR_COTA, stroke_width=2)
        d60 = np.array([np.cos(PI / 3), np.sin(PI / 3), 0])
        ext14 = DashedLine(Q1 - d60 * 1.5, Q1 + d60 * 4.6, color=COR_COTA, stroke_width=2)
        with self.fala("Agora as forças não estão na mesma reta.") as t:
            self.play(Create(ext14), Indicate(f14, color=COR_TEXTO), run_time=t.duration * 0.6)
            self.play(Indicate(f12, color=COR_TEXTO), run_time=t.duration * 0.4)

        # ---------------- 9. Componentes ----------------
        ponta = Q1 + d60 * ESC * F13
        eixo_y = DashedLine(Q1 + DOWN * 0.8, Q1 + UP * 3.2, color=COR_COTA, stroke_width=2)
        proj_x = DashedLine(ponta, [ponta[0], Q1[1], 0], color=COR_COTA)
        proj_y = DashedLine(ponta, [Q1[0], ponta[1], 0], color=COR_COTA)
        cx = MathTex(r"1{,}025", font_size=28, color=COR_TEXTO).next_to([ponta[0], Q1[1], 0], DOWN, buff=0.15)
        cy = MathTex(r"1{,}775", font_size=28, color=COR_TEXTO).next_to([Q1[0], ponta[1], 0], LEFT, buff=0.15)
        with self.fala("Decompomos F um quatro em x e y.") as t:
            self.play(FadeOut(ext14), Create(eixo_y), run_time=t.duration * 0.3)
            self.play(Create(proj_x), Create(proj_y), Write(cx), Write(cy), run_time=t.duration * 0.7)

        # ---------------- 10 e 11. Somas na caixa ----------------
        caixa = caixa_preta(UR, 6.2, 1.5).shift(DOWN * 0.9)
        sx = MathTex(r"F_x = -1{,}15 + 1{,}025 = -1{,}25\times10^{-25}\ \mathrm{N}", font_size=30)
        sy = MathTex(r"F_y = 0 + 1{,}775 = 1{,}78\times10^{-24}\ \mathrm{N}", font_size=30)
        for m in (sx, sy):
            m.move_to(caixa).set_opacity(0)
        with self.fala("Soma em x: menos 1,25 vezes dez a menos vinte e cinco.") as t:
            self.play(FadeOut(eq), FadeIn(caixa), run_time=0.6)
            self.add(sx)
            self.play(sx.animate.set_opacity(1), Indicate(cx, color=WHITE), run_time=max(t.duration - 0.6, 1))
        with self.fala("Soma em y: 1,78 vezes dez a menos vinte e quatro.") as t:
            self.add(sy)
            self.play(sx.animate.set_opacity(0), sy.animate.set_opacity(1), Indicate(cy, color=WHITE),
                      run_time=t.duration)

        # ---------------- 12. Resultante ----------------
        fx, fy = -0.125, 1.775
        vet_res = np.array([fx, fy, 0]) * ESC
        res2 = seta(Q1, Q1 + vet_res, cor=COR_RESULTADO, espessura=8)
        a_res = np.arctan2(fy, fx)
        arco94 = Arc(radius=0.55, start_angle=0, angle=a_res, arc_center=Q1, color=COR_RESULTADO, stroke_width=4)
        txt94 = MathTex(r"1{,}78\times10^{-24}\ \mathrm{N},\ 94^\circ", font_size=32, color=COR_RESULTADO)
        txt94.next_to(Q1 + vet_res, UL, buff=0.15).shift(UP * 0.2)
        with self.fala("Resultado: 1,78 vezes dez a menos vinte e quatro newtons, a 94 graus.") as t:
            self.play(sy.animate.set_opacity(0), FadeOut(caixa), FadeOut(VGroup(leit_ang, arco)), run_time=0.5)
            self.play(GrowArrow(res2), Create(arco94), Write(txt94), run_time=max(t.duration - 0.5, 1.2))

        # ---------------- 13. Desenhe antes de calcular ----------------
        errada = seta(Q1, Q1 + rotacao_z(vet_res, PI), cor=COR_ERRO, espessura=6, opacidade=0.8)
        txt86 = MathTex(r"-86^\circ\ ?", font_size=34, color=COR_ERRO).next_to(Q1 + rotacao_z(vet_res, PI), RIGHT)
        dica = Text("Desenhe antes de calcular!", font_size=32, color=COR_TEXTO).to_corner(UR)
        with self.fala("A calculadora diria menos 86 graus. Desenhe antes de calcular.") as t:
            self.play(GrowArrow(errada), Write(txt86), run_time=t.duration * 0.35)
            self.play(Rotate(errada, angle=PI, about_point=Q1), FadeOut(txt86), run_time=t.duration * 0.35)
            self.play(errada.animate.set_color(COR_TEXTO), Write(dica), Indicate(txt94), run_time=t.duration * 0.3)
        self.wait(0.5)

## Cena 9 — Exemplo 3D das notas de Física III (1018 N)

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombExemplo3D
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombExemplo3D(CenaCoulomb):
    def construct(self):
        self.preparar()
        self.set_camera_orientation(phi=60 * DEGREES, theta=-50 * DEGREES, zoom=1.3)

        eixos = ThreeDAxes(x_range=[-1, 4, 1], y_range=[-1, 3, 1], z_range=[-1, 2, 1],
                           x_length=5, y_length=4, z_length=3,
                           axis_config={"color": COR_COTA, "include_tip": True})
        eixos.shift(-eixos.c2p(1.5, 1, 0))
        rx = MathTex("x", color=COR_COTA).move_to(eixos.c2p(4.3, 0, 0))
        ry = MathTex("y", color=COR_COTA).move_to(eixos.c2p(0, 3.3, 0))
        rz = MathTex("z", color=COR_COTA).move_to(eixos.c2p(0, 0, 2.3))
        self.add_fixed_orientation_mobjects(rx, ry, rz)
        self.remove(rx, ry, rz)

        A = 3.0  # a = 1 cm  -> 3 unidades dos eixos
        P = eixos.c2p(A, A / 2, 0)
        P1 = eixos.c2p(A, 0, 0)
        P2 = eixos.c2p(0, 0, 0)
        ESC = 1 / 400  # unidades de tela por newton

        # ---------------- 1. Eixos ----------------
        with self.fala("Agora, um caso no espaço.") as t:
            self.play(Create(eixos), FadeIn(rx), FadeIn(ry), FadeIn(rz), run_time=t.duration)
            self.begin_ambient_camera_rotation(rate=0.05)

        # ---------------- 2. Carga q ----------------
        q = carga_3d(-1, 0.18).move_to(P)
        guias = VGroup(DashedLine(P, P1, color=COR_COTA), DashedLine(P, eixos.c2p(0, A / 2, 0), color=COR_COTA))
        rq = MathTex(r"q = -1\,\mu\mathrm{C}", font_size=30, color=COR_TEXTO)
        self.add_fixed_orientation_mobjects(rq)
        self.remove(rq)
        rq.move_to(P + UP * 0.5 + OUT * 0.5)
        with self.fala("Uma carga q, de menos um microcoulomb, fica em a, a sobre dois.") as t:
            self.play(GrowFromCenter(q), Create(guias), FadeIn(rq), run_time=t.duration * 0.7)

        # ---------------- 3. q1 ----------------
        q1 = carga_3d(+1, 0.15).move_to(P1)
        rq1 = MathTex(r"q_1 = +1\,\mu\mathrm{C}", font_size=30, color=COR_TEXTO)
        self.add_fixed_orientation_mobjects(rq1)
        self.remove(rq1)
        rq1.move_to(P1 + DOWN * 0.9 + LEFT * 0.4)
        with self.fala("q um, de mais um microcoulomb, fica em a sobre o eixo x.") as t:
            self.play(GrowFromCenter(q1), FadeIn(rq1), run_time=t.duration * 0.6)

        # ---------------- 4. q2 ----------------
        q2 = carga_3d(+1, 0.3).move_to(P2)
        rq2 = MathTex(r"q_2 = 5^{3/2}\,\mu\mathrm{C}", font_size=30, color=COR_TEXTO)
        self.add_fixed_orientation_mobjects(rq2)
        self.remove(rq2)
        rq2.move_to(P2 + LEFT * 0.9 + DOWN * 0.5)
        with self.fala("E q dois, de cerca de 11,18 microcoulombs, na origem.") as t:
            self.play(GrowFromCenter(q2), FadeIn(rq2), run_time=t.duration * 0.6)

        # ---------------- 5 e 6. Forças ----------------
        F1 = np.array([0, -360, 0]) * ESC
        F2 = np.array([-720, -360, 0]) * ESC
        f1 = seta(P, P + F1, espessura=8, ponta=0.22)
        f2 = seta(P, P + F2, espessura=8, ponta=0.22)
        with self.fala("q um puxa q para menos y, com 360 newtons.") as t:
            self.play(GrowArrow(f1), run_time=t.duration * 0.6)
        with self.fala("q dois puxa q na diagonal, com 805 newtons.") as t:
            self.play(GrowArrow(f2), run_time=t.duration * 0.6)

        # ---------------- 7. Soma ----------------
        c1 = DashedLine(P + F1, P + F1 + F2, color=COR_COTA)
        c2 = DashedLine(P + F2, P + F1 + F2, color=COR_COTA)
        res = seta(P, P + F1 + F2, cor=COR_RESULTADO, espessura=9, ponta=0.25)
        eq = MathTex(r"\vec{F}_{total} = (-720\,\hat{x} - 720\,\hat{y})\ \mathrm{N}", font_size=38)
        eq.to_corner(UL)
        self.fixar(eq)
        with self.fala("A soma dá menos 720 newtons em x e em y.") as t:
            self.play(Create(c1), Create(c2), run_time=t.duration * 0.4)
            self.play(GrowArrow(res), Write(eq), run_time=t.duration * 0.6)

        # ---------------- 8. 1018 N ----------------
        leitura = VGroup(MathTex(r"|\vec{F}_{total}| \approx", font_size=38),
                         DecimalNumber(0, num_decimal_places=0, unit=r"\ \mathrm{N}", font_size=38, group_with_commas=False))
        leitura.arrange(RIGHT).set_color(COR_RESULTADO).next_to(eq, DOWN, aligned_edge=LEFT)
        cont = ValueTracker(0)
        leitura[1].add_updater(lambda m: m.set_value(cont.get_value()))
        peso = Text("≈ o peso de uns 104 kg", font_size=26, color=COR_TEXTO).next_to(leitura, DOWN, aligned_edge=LEFT)
        self.fixar(leitura, peso)
        with self.fala("O módulo é cerca de 1018 newtons. Enorme!") as t:
            self.add(leitura)
            self.play(cont.animate.set_value(1018), run_time=t.duration * 0.6)
            self.play(FadeIn(peso, shift=UP), run_time=t.duration * 0.3)

        # ---------------- 9. Barras comparativas ----------------
        e = 4.0 / 1165
        b1 = Rectangle(width=1165 * e, height=0.35, color=COR_FORCA, fill_opacity=0.8, stroke_width=0)
        b2 = Rectangle(width=1018 * e, height=0.35, color=COR_RESULTADO, fill_opacity=0.8, stroke_width=0)
        VGroup(b1, b2).arrange(DOWN, buff=0.45, aligned_edge=LEFT).to_corner(DL, buff=0.6).shift(UP * 0.4)
        t1 = MathTex(r"|\vec{F}_1| + |\vec{F}_2| = 1165\ \mathrm{N}", font_size=28).next_to(b1, UP, aligned_edge=LEFT, buff=0.08)
        t2 = MathTex(r"|\vec{F}_1 + \vec{F}_2| = 1018\ \mathrm{N}", font_size=28).next_to(b2, UP, aligned_edge=LEFT, buff=0.08)
        self.fixar(b1, b2, t1, t2)
        with self.fala("Mas 360 mais 805 dá 1165. Por que não bate?") as t:
            self.play(GrowFromEdge(b1, LEFT), GrowFromEdge(b2, LEFT), FadeIn(t1), FadeIn(t2),
                      run_time=t.duration * 0.8)

        # ---------------- 10. Módulo não é linear ----------------
        abre = ValueTracker(70 * DEGREES)
        O = np.array([3.0, -2.4, 0])

        def triangulo():
            a = RIGHT * 1.4
            b = 1.9 * np.array([np.cos(abre.get_value()), np.sin(abre.get_value()), 0])
            g = VGroup(seta(O, O + a, cor=COR_FORCA, espessura=5),
                       seta(O + a, O + a + b, cor=COR_FORCA, espessura=5),
                       seta(O, O + a + b, cor=COR_RESULTADO, espessura=6))
            return g

        tri = always_redraw(triangulo)
        desig = MathTex(r"|\vec{a}+\vec{b}| \le |\vec{a}| + |\vec{b}|", font_size=34, color=COR_TEXTO)
        desig.move_to([4.0, -0.7, 0])
        self.fixar(tri, desig)
        with self.fala("Porque o módulo de um vetor não é uma função linear.") as t:
            self.play(FadeIn(tri), Write(desig), run_time=t.duration * 0.4)
            self.play(abre.animate.set_value(0), run_time=t.duration * 0.4)
            self.play(abre.animate.set_value(70 * DEGREES), Indicate(desig), run_time=t.duration * 0.2)

        # ---------------- 11. Um microcoulomb é enorme ----------------
        uc = MathTex(r"1\,\mu\mathrm{C}", font_size=60, color=COR_TEXTO).to_corner(UR)
        self.fixar(uc)
        with self.fala("E repare: um microcoulomb já é uma carga gigantesca.") as t:
            self.stop_ambient_camera_rotation()
            self.begin_ambient_camera_rotation(rate=0.15)
            self.play(FadeIn(uc, scale=1.5), run_time=t.duration * 0.4)
            self.play(Indicate(uc, scale_factor=1.3), run_time=t.duration * 0.4)
        self.wait(1)

## Cena 10 — Equilíbrio de um próton: Exemplo 21.02

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombEquilibrio
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombEquilibrio(CenaCoulomb):
    def construct(self):
        self.preparar()

        X0 = -4.5          # posição de q1 na tela
        L = 2.4            # L na tela
        Y = -0.8           # altura do eixo
        ESC_F = 0.55       # comprimento de seta por unidade de força
        MAX = 2.0

        def tela(x):
            return np.array([X0 + L * x, Y, 0])

        # ---------------- 1. Cargas fixas ----------------
        eixo = Line([-7, Y, 0], [7, Y, 0], color=COR_COTA, stroke_width=2)
        c1 = carga_2d(+1, 0.38).move_to(tela(0))
        c2 = carga_2d(-1, 0.26).move_to(tela(1))
        r1 = MathTex("+8q", font_size=34, color=COR_TEXTO).next_to(c1, DOWN, buff=0.25)
        r2 = MathTex("-2q", font_size=34, color=COR_TEXTO).next_to(c2, DOWN, buff=0.25)
        cota = DoubleArrow(tela(0) + DOWN * 1.1, tela(1) + DOWN * 1.1, buff=0, color=COR_COTA,
                           stroke_width=2, tip_length=0.15)
        rL = MathTex("L", font_size=32, color=COR_COTA).next_to(cota, DOWN, buff=0.08)
        pergunta = MathTex("?", font_size=80, color=COR_TEXTO).move_to(tela(2) + UP * 1.2)
        with self.fala("Onde um próton fica em equilíbrio entre mais oito q e menos dois q?") as t:
            self.play(Create(eixo), GrowFromCenter(c1), GrowFromCenter(c2), Write(r1), Write(r2),
                      run_time=t.duration * 0.45)
            self.play(GrowFromCenter(cota), Write(rL), FadeIn(pergunta, shift=DOWN), run_time=t.duration * 0.3)
            self.play(pergunta.animate(rate_func=there_and_back).shift(UP * 0.4), run_time=t.duration * 0.25)

        # ---------------- Próton e forças ----------------
        x = ValueTracker(0.5)

        def f1():   # repulsão de q1: sentido = sinal(x)
            xv = x.get_value()
            return np.sign(xv) * 8 / max(xv ** 2, 0.05)

        def f2():   # atração para q2: sentido = sinal(1 - x)
            xv = x.get_value()
            return np.sign(1 - xv) * 2 / max((xv - 1) ** 2, 0.05)

        def fres():
            return f1() + f2()

        def lim(v):
            return float(np.clip(ESC_F * v, -MAX, MAX))

        proton = always_redraw(lambda: carga_2d(+1, 0.17).move_to(tela(x.get_value())))
        rp = always_redraw(lambda: MathTex("p", font_size=28, color=COR_TEXTO)
                           .next_to(tela(x.get_value()), DOWN, buff=0.25))
        s1 = always_redraw(lambda: seta(tela(x.get_value()) + UP * 0.32,
                                        tela(x.get_value()) + UP * 0.32 + RIGHT * lim(f1()), espessura=5))
        s2 = always_redraw(lambda: seta(tela(x.get_value()) + UP * 0.6,
                                        tela(x.get_value()) + UP * 0.6 + RIGHT * lim(f2()),
                                        cor=COR_CARGA_NEG, espessura=5))
        sr = always_redraw(lambda: seta(tela(x.get_value()) + UP * 0.95,
                                        tela(x.get_value()) + UP * 0.95 + RIGHT * lim(fres()),
                                        cor=COR_RESULTADO, espessura=7))
        legenda = VGroup(
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=COR_FORCA, stroke_width=5), MathTex(r"\vec{F}_1", font_size=30)),
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=COR_CARGA_NEG, stroke_width=5), MathTex(r"\vec{F}_2", font_size=30)),
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=COR_RESULTADO, stroke_width=7), Text("resultante", font_size=22)),
        )
        for g in legenda:
            g.arrange(RIGHT, buff=0.15)
        legenda.arrange(DOWN, aligned_edge=LEFT, buff=0.15).to_corner(UL)
        with self.fala("Vamos arrastar o próton pelo eixo.") as t:
            self.play(FadeOut(pergunta), GrowFromCenter(proton), FadeIn(rp), FadeIn(legenda), run_time=0.8)
            self.add(s1, s2, sr)
            self.play(x.animate.set_value(0.65), run_time=max(t.duration - 0.8, 1))

        # ---------------- Regiões ----------------
        def faixa(xa, xb, nome):
            a, b = tela(xa), tela(xb)
            r = Rectangle(width=abs(b[0] - a[0]), height=3.4, stroke_width=0, fill_color=COR_ERRO,
                          fill_opacity=0.12).move_to([(a[0] + b[0]) / 2, Y + 0.6, 0])
            t_ = Text(nome, font_size=26, color=COR_GAUSS).next_to(r, UP, buff=-0.45)
            return VGroup(r, t_)

        regP = faixa(0.12, 0.88, "P")
        regS = faixa(-1.0, -0.12, "S")

        # ---------------- 3. Região P ----------------
        with self.fala("Entre as cargas, as duas forças apontam para o mesmo lado.") as t:
            self.play(FadeIn(regP), run_time=0.6)
            self.play(x.animate.set_value(0.25), run_time=max((t.duration - 0.6) / 2, 0.8))
            self.play(x.animate.set_value(0.8), run_time=max((t.duration - 0.6) / 2, 0.8))

        # ---------------- 4. Região S ----------------
        with self.fala("À esquerda, F um sempre vence: carga maior e mais perto.") as t:
            self.play(x.animate.set_value(-0.55), run_time=t.duration * 0.5, rate_func=smooth)
            self.play(FadeIn(regS), x.animate.set_value(-0.9), run_time=t.duration * 0.5)

        # ---------------- 5. Região R + gráfico ----------------
        eixos = Axes(x_range=[0, 4.2, 1], y_range=[-2, 2, 1], x_length=4.2, y_length=2.4,
                     axis_config={"color": COR_COTA, "include_tip": False}).to_corner(UR, buff=0.35)
        rx = MathTex("x/L", font_size=24, color=COR_COTA).next_to(eixos.x_axis, DOWN, buff=0.05).align_to(eixos, RIGHT)
        rg = Text("força resultante", font_size=20, color=COR_COTA).next_to(eixos, UP, buff=0.05)
        ponto = always_redraw(lambda: Dot(eixos.c2p(np.clip(x.get_value(), 0, 4.2),
                                                    np.clip(fres(), -2, 2)), radius=0.06,
                                          color=COR_RESULTADO))
        regR = faixa(1.12, 4.6, "R")
        regR[0].set_fill(COR_GAUSS, opacity=0.03)
        with self.fala("À direita de q dois, as forças podem se cancelar.") as t:
            self.play(FadeOut(VGroup(regP, regS)), x.animate.set_value(1.2), run_time=t.duration * 0.35)
            self.play(FadeIn(regR), Create(eixos), FadeIn(rx), FadeIn(rg), run_time=t.duration * 0.2)
            self.add(ponto)
            rastro = TracedPath(ponto.get_center, stroke_color=COR_RESULTADO, stroke_width=3)
            self.add(rastro)
            self.play(x.animate.set_value(4.0), run_time=t.duration * 0.45)

        # ---------------- 6. Igualando ----------------
        caixa = caixa_preta(DL, 6.6, 1.7)
        eq = MathTex(r"\frac{1}{4\pi\varepsilon_0}\frac{8q\,q_p}{x^2} = \frac{1}{4\pi\varepsilon_0}\frac{2q\,q_p}{(x-L)^2}",
                     font_size=34).move_to(caixa)
        with self.fala("Igualando os módulos das duas forças...") as t:
            self.play(FadeIn(caixa), Write(eq), x.animate.set_value(3.0), run_time=t.duration)

        # ---------------- 7. x = 2L ----------------
        resp = MathTex(r"x = 2L", font_size=48, color=COR_RESULTADO).move_to(caixa)
        moldura = SurroundingRectangle(resp, color=YELLOW, buff=0.2)
        with self.fala("...encontramos x igual a dois L.") as t:
            self.play(x.animate.set_value(2.0), run_time=t.duration * 0.5)
            self.play(ReplacementTransform(eq, resp), Create(moldura), Flash(tela(2), color=COR_TEXTO),
                      run_time=t.duration * 0.5)

        # ---------------- 8. Instável para a esquerda ----------------
        with self.fala("Mas esse equilíbrio é instável.") as t:
            rastro.clear_updaters()
            self.play(x.animate.set_value(1.93), run_time=0.5)
            self.play(x.animate.set_value(1.12), run_time=max(t.duration - 0.5, 1), rate_func=rush_into)

        # ---------------- 9. Instável para a direita ----------------
        with self.fala("Um empurrãozinho, e ele foge.") as t:
            self.play(x.animate.set_value(2.0), run_time=0.6)
            self.play(x.animate.set_value(2.07), run_time=0.4)
            self.play(x.animate.set_value(5.2), run_time=max(t.duration - 1.0, 1), rate_func=rush_into)
        self.wait(0.5)

## Cena 11 — Teoremas das cascas e esferas condutoras

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombCascas
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombCascas(CenaCoulomb):
    def construct(self):
        self.preparar()
        rng = np.random.default_rng(21)
        self.set_camera_orientation(phi=70 * DEGREES, theta=-60 * DEGREES, zoom=1.0)
        self.begin_ambient_camera_rotation(rate=0.04)

        RAIO = 1.5
        casca = Sphere(radius=RAIO, resolution=(12, 18))
        casca.set_style(fill_color=COR_GAUSS, fill_opacity=0.12, stroke_color=COR_GAUSS,
                        stroke_width=0.4, stroke_opacity=0.2)
        sinais = VGroup(*[Dot3D(p, radius=0.045, color=COR_CARGA, resolution=(4, 4))
                          for p in pontos_fibonacci(60, RAIO)])
        conjunto = VGroup(casca, sinais)
        conjunto.add_updater(lambda m, dt: m.rotate(0.3 * dt, axis=OUT, about_point=ORIGIN))

        # ---------------- 1. Casca ----------------
        with self.fala("Uma casca esférica com carga uniforme tem um truque.") as t:
            self.play(Create(casca), run_time=t.duration * 0.5)
            self.play(LaggedStart(*[FadeIn(s, scale=3) for s in sinais], lag_ratio=0.02),
                      run_time=t.duration * 0.5)
            self.add(conjunto)

        # ---------------- 2. Partícula de fora ----------------
        orb = ValueTracker(0.0)
        raio_orb = ValueTracker(3.0)
        mag = ValueTracker(1.0)

        def pos_teste():
            a = orb.get_value()
            return raio_orb.get_value() * np.array([np.cos(a), np.sin(a), 0])

        teste = always_redraw(lambda: carga_3d(+1, 0.12, resolucao=(6, 6)).move_to(pos_teste()))
        f_teste = always_redraw(lambda: seta_direcao(pos_teste(), pos_teste(),
                                                     1.6 * mag.get_value() * 9 / max(raio_orb.get_value(), 1.5) ** 2,
                                                     espessura=7, ponta=0.22))
        pontual = carga_3d(+1, 0.18, resolucao=(8, 8))
        with self.fala("Para quem está fora, é como se toda a carga estivesse no centro.") as t:
            self.add(teste, f_teste)
            self.play(orb.animate.set_value(PI), run_time=t.duration * 0.4)
            conjunto.clear_updaters()
            self.play(FadeOut(conjunto), FadeIn(pontual, scale=0.2), orb.animate.set_value(1.4 * PI),
                      run_time=t.duration * 0.3)
            self.play(FadeIn(conjunto), FadeOut(pontual), orb.animate.set_value(1.8 * PI),
                      run_time=t.duration * 0.3)
            conjunto.add_updater(lambda m, dt: m.rotate(0.3 * dt, axis=OUT, about_point=ORIGIN))

        # ---------------- 3. Partícula de dentro ----------------
        def parciais():
            p = pos_teste()
            g = VGroup()
            for k in range(8):
                a = k * TAU / 8
                d = np.array([np.cos(a), np.sin(a), 0])
                g.add(seta_direcao(p, d, 0.5, cor=COR_FORCA, opacidade=0.5, espessura=4))
            return g

        with self.fala("Para quem está dentro, a casca não exerce força.") as t:
            self.play(raio_orb.animate.set_value(0.6), mag.animate.set_value(0), run_time=t.duration * 0.4)
            pequenas = parciais()
            self.play(LaggedStart(*[GrowArrow(s) for s in pequenas], lag_ratio=0.08), run_time=t.duration * 0.3)
            self.play(pequenas.animate.set_color(COR_ERRO), run_time=t.duration * 0.1)
            self.play(FadeOut(pequenas, scale=0.3), run_time=t.duration * 0.2)

        # ---------------- 4. Condutor: carga vai para a superfície ----------------
        eletrons = VGroup(*[Dot3D(rng.uniform(-0.3, 0.3, 3), radius=0.06, color=COR_CARGA_NEG,
                                  resolution=(4, 4)) for _ in range(40)])
        destinos = pontos_fibonacci(40, RAIO)
        with self.fala("Num condutor esférico, a carga extra vai para a superfície.") as t:
            self.play(FadeOut(sinais), FadeOut(teste), FadeOut(f_teste), run_time=0.6)
            self.play(FadeIn(eletrons, scale=0.5), run_time=0.5)
            self.play(LaggedStart(*[e.animate.move_to(d) for e, d in zip(eletrons, destinos)], lag_ratio=0.02),
                      run_time=max(t.duration - 1.1, 1.2), rate_func=rate_functions.ease_out_cubic)

        # ---------------- 5. Esferas A e B (2D) ----------------
        A = np.array([-2.5, 0.3, 0])
        B = np.array([2.5, 0.3, 0])
        cor_a = ValueTracker(1.0)
        cor_b = ValueTracker(0.0)
        esf_a = always_redraw(lambda: Circle(radius=0.6, stroke_width=0, fill_opacity=1,
                                             fill_color=interpolate_color(COR_NEUTRA, COR_CARGA, cor_a.get_value())).move_to(A))
        esf_b = always_redraw(lambda: Circle(radius=0.6, stroke_width=0, fill_opacity=1,
                                             fill_color=interpolate_color(COR_NEUTRA, COR_CARGA, cor_b.get_value())).move_to(B))
        rot_a = MathTex(r"A:\ +Q", font_size=36, color=COR_TEXTO).next_to(A, UP, buff=0.8)
        rot_b = MathTex(r"B:\ 0", font_size=36, color=COR_TEXTO).next_to(B, UP, buff=0.8)
        cota = DoubleArrow(A + DOWN * 1.0, B + DOWN * 1.0, buff=0, color=COR_COTA, stroke_width=2, tip_length=0.15)
        ra = MathTex("a", font_size=32, color=COR_COTA).next_to(cota, DOWN, buff=0.08)
        with self.fala("Duas esferas condutoras iguais: A com mais Q, B neutra.") as t:
            self.stop_ambient_camera_rotation()
            conjunto.clear_updaters()
            self.play(FadeOut(VGroup(casca, eletrons)), run_time=0.6)
            self.move_camera(phi=0, theta=-90 * DEGREES, zoom=1.0, run_time=0.8)
            self.play(FadeIn(esf_a), FadeIn(esf_b), Write(rot_a), Write(rot_b), GrowFromCenter(cota), Write(ra),
                      run_time=max(t.duration - 1.4, 1))

        # ---------------- 6. Fio ----------------
        caminho = ArcBetweenPoints(B + DOWN * 0.6, A + DOWN * 0.6, angle=-PI / 3)
        fio = caminho.copy().set_stroke(COR_COBRE, width=5)
        moveis = VGroup(*[carga_2d(-1, 0.1) for _ in range(6)])
        with self.fala("Ligadas por um fio, elétrons passam de B para A.") as t:
            self.play(FadeOut(VGroup(cota, ra)), Create(fio), run_time=t.duration * 0.3)
            for e in moveis:
                e.move_to(B + DOWN * 0.6)
            self.add(moveis)
            self.play(LaggedStart(*[MoveAlongPath(e, caminho) for e in moveis], lag_ratio=0.15),
                      cor_b.animate.set_value(0.6), cor_a.animate.set_value(0.6), run_time=t.duration * 0.6)
            self.play(FadeOut(moveis), run_time=0.2)

        # ---------------- 7. Q/2 cada ----------------
        rot_a2 = MathTex(r"A:\ +Q/2", font_size=36, color=COR_TEXTO).move_to(rot_a)
        rot_b2 = MathTex(r"B:\ +Q/2", font_size=36, color=COR_TEXTO).move_to(rot_b)
        rep_a = seta(A + LEFT * 0.65, A + LEFT * 1.8)
        rep_b = seta(B + RIGHT * 0.65, B + RIGHT * 1.8)
        eqf = MathTex(r"F = \frac{1}{4\pi\varepsilon_0}\frac{(Q/2)(Q/2)}{a^2} = \frac{1}{16\pi\varepsilon_0}\left(\frac{Q}{a}\right)^2",
                      font_size=38, color=COR_RESULTADO).to_edge(UP, buff=0.4)
        with self.fala("Cada uma fica com mais Q sobre dois, e elas se repelem.") as t:
            self.play(FadeOut(fio), Transform(rot_a, rot_a2), Transform(rot_b, rot_b2), run_time=t.duration * 0.35)
            self.play(GrowArrow(rep_a), GrowArrow(rep_b), Write(eqf), run_time=t.duration * 0.65)

        # ---------------- 8. Terra ----------------
        terra_p = A + DOWN * 2.6
        terra = VGroup(Line(A + DOWN * 0.6, terra_p, color=COR_COBRE, stroke_width=4),
                       Line(terra_p + LEFT * 0.5, terra_p + RIGHT * 0.5, color=COR_GAUSS, stroke_width=4),
                       Line(terra_p + DOWN * 0.15 + LEFT * 0.33, terra_p + DOWN * 0.15 + RIGHT * 0.33, color=COR_GAUSS, stroke_width=4),
                       Line(terra_p + DOWN * 0.3 + LEFT * 0.16, terra_p + DOWN * 0.3 + RIGHT * 0.16, color=COR_GAUSS, stroke_width=4))
        sobe = VGroup(*[carga_2d(-1, 0.1).move_to(terra_p) for _ in range(4)])
        rot_a3 = MathTex(r"A:\ q = 0", font_size=36, color=COR_TEXTO).move_to(rot_a)
        with self.fala("Ligando A à terra, ela se neutraliza.") as t:
            self.play(Create(terra), run_time=t.duration * 0.3)
            self.add(sobe)
            self.play(LaggedStart(*[e.animate.move_to(A) for e in sobe], lag_ratio=0.2),
                      cor_a.animate.set_value(0), run_time=t.duration * 0.4)
            self.play(FadeOut(sobe), run_time=0.2)
            self.play(Transform(rot_a, rot_a3), FadeOut(VGroup(rep_a, rep_b)), eqf.animate.set_opacity(0.25),
                      run_time=t.duration * 0.3)
        self.wait(0.5)

## Cena 12 — Carga quantizada e conservada

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombQuantizacao
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombQuantizacao(CenaCoulomb):
    def construct(self):
        self.preparar()

        # ---------------- 1. Fluido -> pacotes ----------------
        barra = Rectangle(width=10, height=0.6, stroke_width=0, fill_color=COR_CARGA, fill_opacity=0.7)
        barra.move_to(UP * 1.5)
        pacotes = VGroup(*[Square(0.42, stroke_width=0, fill_color=COR_CARGA, fill_opacity=1) for _ in range(20)])
        pacotes.arrange(RIGHT, buff=0.08).move_to(barra)
        with self.fala("A carga não é um fluido contínuo. Vem em pacotes.") as t:
            self.play(GrowFromEdge(barra, LEFT), run_time=t.duration * 0.35)
            self.play(LaggedStart(*[GrowFromCenter(p) for p in pacotes], lag_ratio=0.08),
                      barra.animate.set_opacity(0), run_time=t.duration * 0.65)
            self.remove(barra)

        # ---------------- 2. q = n e ----------------
        eq = MathTex(r"q = n\,e,\quad n = \pm1, \pm2, \pm3, \dots", font_size=48).move_to(UP * 0.2)
        n = ValueTracker(0)
        cont = VGroup(MathTex("n =", font_size=40), Integer(0, font_size=40)).arrange(RIGHT)
        cont.set_color(COR_TEXTO).next_to(eq, DOWN, buff=0.4)
        cont[1].add_updater(lambda m: m.set_value(int(round(n.get_value()))))
        bolinhas = VGroup(*[carga_2d(+1, 0.2) for _ in range(8)]).arrange(RIGHT, buff=0.25).move_to(DOWN * 1.9)
        with self.fala("Toda carga é um múltiplo inteiro de e.") as t:
            self.play(pacotes.animate.scale(0.6).to_edge(UP, buff=0.3), Write(eq), FadeIn(cont),
                      run_time=t.duration * 0.4)
            passo = t.duration * 0.6 / 8
            for k in range(8):
                self.play(n.animate.set_value(k + 1), GrowFromCenter(bolinhas[k]), run_time=passo)

        # ---------------- 3. Valor de e ----------------
        caixa = caixa_preta(UR, 5.4, 1.2).shift(DOWN * 1.2)
        val_e = MathTex(r"e = 1{,}602\times10^{-19}\ \mathrm{C}", font_size=40, color=COR_TEXTO).move_to(caixa)
        with self.fala("A carga elementar vale cerca de 1,602 vezes dez a menos dezenove coulombs.") as t:
            self.play(FadeIn(caixa), Write(val_e), run_time=t.duration * 0.6)
            self.play(Indicate(val_e), run_time=t.duration * 0.4)

        # ---------------- 4. 3,57 e ----------------
        errado = MathTex(r"q = 3{,}57\,e", font_size=52, color=COR_ERRO).move_to(DOWN * 3.0 + LEFT * 3)
        with self.fala("Não existe carga de 3,57 e.") as t:
            self.play(FadeIn(errado, shift=UP), run_time=t.duration * 0.4)
            self.play(Create(Cross(errado, stroke_color=RED, stroke_width=6)), run_time=t.duration * 0.6)

        # ---------------- 5. Placar de conservação ----------------
        placar_tit = Text("carga total", font_size=26, color=COR_GAUSS)
        placar_val = MathTex(r"+92e", font_size=40, color=COR_TEXTO)
        placar = VGroup(placar_tit, placar_val).arrange(DOWN, buff=0.1).to_corner(UR)
        moldura = SurroundingRectangle(placar, color=COR_COTA, buff=0.15)
        with self.fala("E a carga total de um sistema isolado se conserva.") as t:
            self.play(*[FadeOut(m) for m in self.mobjects if m is not self.relogio], run_time=0.8)
            self.play(FadeIn(placar), Create(moldura), run_time=max(t.duration - 0.8, 1))

        # ---------------- 6. Decaimento alfa ----------------
        uranio = VGroup(Circle(radius=1.0, fill_color=COR_CARGA, fill_opacity=0.85, stroke_width=0),
                        MathTex(r"^{238}\mathrm{U}", font_size=44))
        uranio.add_updater(tremor(0.04, 9, direcao=RIGHT))
        torio = VGroup(Circle(radius=0.95, fill_color=COR_CARGA, fill_opacity=0.85, stroke_width=0),
                       MathTex(r"^{234}\mathrm{Th}", font_size=40))
        alfa = VGroup(Circle(radius=0.32, fill_color=COR_CARGA, fill_opacity=1, stroke_width=0),
                      MathTex(r"^{4}\mathrm{He}", font_size=26))
        torio.move_to(ORIGIN)
        alfa.move_to(ORIGIN)
        r_th = MathTex("+90e", font_size=34, color=COR_TEXTO)
        r_he = MathTex("+2e", font_size=34, color=COR_TEXTO)
        balanco = MathTex(r"+92e = +90e + 2e", font_size=40, color=COR_TEXTO).next_to(placar, DOWN, buff=0.4)
        balanco.align_to(moldura, RIGHT)
        with self.fala("O urânio 238 vira tório 234 e uma partícula alfa.") as t:
            self.play(GrowFromCenter(uranio), run_time=t.duration * 0.3)
            uranio.clear_updaters()
            self.remove(uranio)
            self.add(torio, alfa)
            self.play(torio.animate.shift(LEFT * 2.2), alfa.animate.shift(RIGHT * 4.5 + UP * 0.6),
                      run_time=t.duration * 0.4, rate_func=rate_functions.ease_out_cubic)
            r_th.next_to(torio, DOWN); r_he.next_to(alfa, DOWN)
            self.play(FadeIn(r_th), FadeIn(r_he), Write(balanco), Indicate(placar_val),
                      run_time=t.duration * 0.3)

        # ---------------- 7. Aniquilação ----------------
        placar_zero = MathTex(r"0", font_size=40, color=COR_TEXTO).move_to(placar_val)
        eme = carga_2d(-1, 0.25).move_to(LEFT * 5 + DOWN * 1)
        epl = carga_2d(+1, 0.25).move_to(RIGHT * 5 + DOWN * 1)
        r_eme = MathTex("e^-", font_size=32, color=COR_TEXTO).add_updater(lambda m: m.next_to(eme, UP, buff=0.15))
        r_epl = MathTex("e^+", font_size=32, color=COR_TEXTO).add_updater(lambda m: m.next_to(epl, UP, buff=0.15))
        prog = ValueTracker(0)

        def foton(sentido):
            def f():
                p = prog.get_value()
                if p <= 0:
                    return VectorizedPoint(DOWN)
                x0 = DOWN * 1 + sentido * RIGHT * 6 * p
                return ParametricFunction(
                    lambda s: x0 - sentido * RIGHT * s + UP * 0.25 * np.sin(14 * s - 25 * p),
                    t_range=[0, min(1.8, 6 * p), 0.02], color=COR_FORCA, stroke_width=3)
            return f

        g1 = always_redraw(foton(1))
        g2 = always_redraw(foton(-1))
        with self.fala("Um elétron e um pósitron se aniquilam em dois raios gama.") as t:
            self.play(FadeOut(VGroup(torio, alfa, r_th, r_he, balanco)), Transform(placar_val, placar_zero),
                      FadeIn(eme), FadeIn(epl), FadeIn(r_eme), FadeIn(r_epl), run_time=t.duration * 0.25)
            self.play(eme.animate.move_to(DOWN * 1 + LEFT * 0.2), epl.animate.move_to(DOWN * 1 + RIGHT * 0.2),
                      run_time=t.duration * 0.3, rate_func=rush_into)
            self.remove(eme, epl, r_eme, r_epl)
            self.add(g1, g2)
            self.play(Flash(DOWN * 1, color=WHITE, flash_radius=0.6), prog.animate.set_value(1),
                      Indicate(placar_val), run_time=t.duration * 0.45, rate_func=linear)

        # ---------------- 8. Produção de par ----------------
        prog2 = ValueTracker(0)
        V = DOWN * 0.2

        def raio_gama():
            p = prog2.get_value()
            y_top = -3.6 + 3.4 * min(p, 1)
            return ParametricFunction(lambda s: np.array([0.18 * np.sin(12 * s - 20 * p), s, 0]),
                                      t_range=[-3.6, max(y_top, -3.55), 0.02], color=COR_FORCA, stroke_width=3)

        gama = always_redraw(raio_gama)
        trilha_e = ParametricFunction(lambda s: V + np.array([-1.6 * s ** 1.4 - 0.3 * np.sin(3 * s) * s, 3.2 * s, 0]),
                                      t_range=[0, 1, 0.02], color=COR_CARGA_NEG, stroke_width=4)
        trilha_p = ParametricFunction(lambda s: V + np.array([1.6 * s ** 1.4 + 0.3 * np.sin(3 * s) * s, 3.2 * s, 0]),
                                      t_range=[0, 1, 0.02], color=COR_CARGA, stroke_width=4)
        d_e = carga_2d(-1, 0.16).move_to(V)
        d_p = carga_2d(+1, 0.16).move_to(V)
        conta = MathTex(r"0 \;\to\; (-e) + (+e) = 0", font_size=36, color=COR_TEXTO).next_to(moldura, DOWN, buff=0.4)
        conta.align_to(moldura, RIGHT)
        with self.fala("E o contrário: um raio gama cria um par.") as t:
            self.play(FadeOut(VGroup(g1, g2)), run_time=0.4)
            self.add(gama)
            self.play(prog2.animate.set_value(1), run_time=max(t.duration * 0.4, 1), rate_func=linear)
            self.remove(gama)
            self.add(d_e, d_p)
            self.play(Create(trilha_e), Create(trilha_p), MoveAlongPath(d_e, trilha_e), MoveAlongPath(d_p, trilha_p),
                      Write(conta), run_time=max(t.duration * 0.6 - 0.4, 1.2))
        self.wait(0.8)

## Cena 13 — Fechamento e ponte para a Lei de Gauss

In [ ]:
%%manim -pql --fps 15 -v WARNING CoulombFechamento
import importlib, coulomb_comum
importlib.reload(coulomb_comum)
from coulomb_comum import *


class CoulombFechamento(CenaCoulomb):
    def construct(self):
        self.preparar()

        final = MathTex(r"\vec{F} = \frac{1}{4\pi\varepsilon_0}\,", r"q_1 q_2", r"\,\frac{1}{r^2}", r"\,\hat{r}",
                        font_size=60, color=COR_RESULTADO).to_edge(UP, buff=0.6)

        # Par de cargas (q2 pode trocar de sinal)
        q2 = ValueTracker(1.0)
        P1, P2 = np.array([-1.5, -1.2, 0]), np.array([1.5, -1.2, 0])
        c1 = carga_2d(+1, 0.3).move_to(P1)
        c1.add_updater(tremor(0.04, 3, direcao=UP))
        c2 = always_redraw(lambda: carga_2d(1 if q2.get_value() >= 0 else -1, 0.3,
                                            cor=interpolate_color(COR_CARGA_NEG, COR_CARGA, (q2.get_value() + 1) / 2))
                           .move_to(P2 + UP * 0.04 * np.sin(3 * self.t() + 1)))
        f1 = always_redraw(lambda: seta_direcao(c1.get_center(), LEFT, 1.2 * q2.get_value()))
        f2 = always_redraw(lambda: seta_direcao(c2.get_center(), RIGHT, 1.2 * q2.get_value()))

        # ---------------- 1. Recapitulando ----------------
        with self.fala("Recapitulando: duas cargas pontuais em repouso...") as t:
            self.play(FadeIn(c1), FadeIn(c2), run_time=0.8)
            self.play(Write(final), run_time=max(t.duration - 0.8, 1.2))

        # ---------------- 2. Reta que as une ----------------
        with self.fala("...se atraem ou se repelem na reta que as une.") as t:
            self.add(f1, f2)
            self.play(Indicate(final[3], color=COR_TEXTO, scale_factor=1.5), run_time=t.duration * 0.3)
            self.play(q2.animate.set_value(-1), run_time=t.duration * 0.35)
            self.play(q2.animate.set_value(1), run_time=t.duration * 0.35)

        # ---------------- 3. Produto e 1/r² ----------------
        chaves = VGroup()
        for i, txt in ((1, "produto das cargas"), (2, "inverso do quadrado")):
            b = Brace(final[i], DOWN, color=COR_COTA)
            chaves.add(VGroup(b, Text(txt, font_size=20, color=COR_COTA).next_to(b, DOWN, buff=0.08 if i == 1 else 0.45)))
        with self.fala("A força é proporcional ao produto das cargas e cai com um sobre r ao quadrado.") as t:
            self.play(LaggedStart(*[GrowFromCenter(c) for c in chaves], lag_ratio=0.4), run_time=t.duration * 0.6)
            self.play(FadeOut(chaves), run_time=t.duration * 0.3)

        # ---------------- 4. Superposição ----------------
        outras = [(np.array([-3.4, 0.4, 0]), -1), (np.array([-3.0, -3.0, 0]), +1), (np.array([-0.2, -3.2, 0]), -1)]
        cargas_extra = VGroup(*[carga_2d(q, 0.25).move_to(p) for p, q in outras])
        # força em q1 (+): repulsão (sentido P1 - p) se q > 0, atração se q < 0
        forcas = [normalize(P1 - p) * q * 4.0 / max(np.linalg.norm(P1 - p), 1) for p, q in outras]
        setas_ind = VGroup(*[seta(P1, P1 + f, espessura=4) for f in forcas])
        moldura = SurroundingRectangle(final, color=YELLOW, buff=0.25)
        with self.fala("Com várias cargas, somamos vetorialmente.") as t:
            c1.clear_updaters()
            self.play(FadeOut(VGroup(f1, f2, c2)), LaggedStart(*[GrowFromCenter(c) for c in cargas_extra], lag_ratio=0.2),
                      run_time=t.duration * 0.3)
            self.play(LaggedStart(*[GrowArrow(s) for s in setas_ind], lag_ratio=0.2), run_time=t.duration * 0.25)
            # cabeça com cauda
            acumulado = P1.copy()
            anims = []
            for s, f in zip(setas_ind, forcas):
                anims.append(s.animate.shift(acumulado - P1))
                acumulado = acumulado + f
            self.play(*anims, run_time=t.duration * 0.25)
            res = seta(P1, acumulado, cor=COR_RESULTADO, espessura=8)
            self.play(GrowArrow(res), Create(moldura), run_time=t.duration * 0.2)

        # ---------------- 5. Linha de cargas (notebook da Lei de Gauss) ----------------
        num_cargas, raio_carga, espacamento = 17, 0.25, 0.50
        inicio_x = -(num_cargas - 1) * espacamento / 2
        angulos = [PI / 2 - i * (TAU / 10) for i in range(10)]

        def carga_com_linhas(i, inicio):
            pos = np.array([inicio + i * espacamento, -1.2, 0])
            c = carga_2d(+1, raio_carga).move_to(pos)
            linhas = VGroup(*[Arrow(pos + np.array([np.cos(a), np.sin(a), 0]) * (raio_carga + 0.05),
                                    pos + np.array([np.cos(a), np.sin(a), 0]) * (raio_carga + 0.5),
                                    buff=0, color=SOLARIZED_BLUE, tip_length=0.1, stroke_width=3)
                              for a in angulos])
            return c, linhas

        fio = VGroup(); linhas_fio = VGroup()
        for i in range(num_cargas):
            c, l = carga_com_linhas(i, inicio_x)
            fio.add(c); linhas_fio.add(l)
        with self.fala("Mas e quando são milhões de cargas, num fio?") as t:
            self.play(FadeOut(VGroup(c1, cargas_extra, setas_ind, res)), run_time=0.6)
            self.play(LaggedStart(*[GrowFromCenter(c) for c in fio], lag_ratio=0.06), run_time=(t.duration - 0.6) * 0.5)
            self.play(LaggedStart(*[LaggedStart(*[GrowArrow(a) for a in l], lag_ratio=0.05) for l in linhas_fio],
                                  lag_ratio=0.1), run_time=(t.duration - 0.6) * 0.5)

        # ---------------- 6. Contador explode ----------------
        contador = VGroup(Text("cargas:", font_size=30, color=COR_TEXTO), Integer(17, font_size=36, color=COR_TEXTO, group_with_commas=False))
        contador.arrange(RIGHT).to_corner(DL)
        n = ValueTracker(17)
        contador[1].add_updater(lambda m: m.set_value(int(n.get_value())))
        novo_num = 41
        novo_inicio = -(novo_num - 1) * espacamento / 2
        extras = VGroup()
        for i in list(range(0, 12)) + list(range(29, 41)):
            c, l = carga_com_linhas(i, novo_inicio)
            extras.add(VGroup(c, l))
        with self.fala("Somar carga por carga fica impossível.") as t:
            self.play(FadeIn(contador), run_time=0.4)
            self.play(LaggedStart(*[FadeIn(e, scale=0.5) for e in extras], lag_ratio=0.05),
                      n.animate(rate_func=rate_functions.ease_in_expo).set_value(10 ** 6),
                      run_time=max(t.duration - 0.4, 1.5))

        # ---------------- 7. Para Gauss ----------------
        cilindro = Cylinder(radius=raio_carga + 0.02, height=(novo_num - 1) * espacamento + 2 * raio_carga,
                            direction=RIGHT, resolution=(16, 24)).set_style(
            fill_color=COR_CARGA, fill_opacity=1, stroke_width=0, sheen_factor=0.0)
        cilindro.move_to([0, -1.2, 0])
        prox = Text("Próximo: campo elétrico e Lei de Gauss", font_size=34, color=COR_TEXTO)
        self.fixar(prox)
        prox.to_edge(DOWN, buff=0.5)
        with self.fala("Para isso, vamos precisar do campo elétrico e da Lei de Gauss.") as t:
            self.play(FadeOut(VGroup(contador, moldura)), final.animate.scale(0.7).to_corner(UL), run_time=0.6)
            self.add_fixed_in_frame_mobjects(final)
            self.move_camera(phi=50 * DEGREES, theta=-60 * DEGREES, frame_center=[0, -1.2, 0], run_time=t.duration * 0.4,
                             added_anims=[FadeIn(cilindro), FadeOut(VGroup(fio, linhas_fio, extras))])
            self.play(Write(prox), run_time=max(t.duration * 0.6 - 0.6, 1))
        self.begin_ambient_camera_rotation(rate=0.1)
        self.wait(1.5)
        self.play(*[FadeOut(m) for m in self.mobjects if m is not self.relogio], run_time=2)

## Juntar todas as cenas em um único vídeo

Procura o vídeo mais recente de cada cena dentro de `media/` e concatena na ordem do roteiro. Cenas que ainda não foram renderizadas são puladas.

In [ ]:
import glob, os, subprocess

ORDEM = ["CoulombAbertura", "CoulombCondutores", "CoulombBalanca", "CoulombPropriedades", "CoulombFormula", "CoulombVetorial", "CoulombNucleo", "CoulombSuperposicao", "CoulombExemplo3D", "CoulombEquilibrio", "CoulombCascas", "CoulombQuantizacao", "CoulombFechamento"]

videos = []
for cena in ORDEM:
    candidatos = [p for p in glob.glob(f"media/**/{cena}*.mp4", recursive=True)
                  if "partial_movie_files" not in p]
    if not candidatos:
        print(f"(pulando {cena}: ainda não renderizada)")
        continue
    videos.append(max(candidatos, key=os.path.getmtime))

with open("lista.txt", "w") as f:
    for v in videos:
        f.write(f"file '{os.path.abspath(v)}'\n")

subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                "-i", "lista.txt", "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-c:a", "aac", "-b:a", "160k", "lei_de_coulomb_completo.mp4"], check=True)
print(f"{len(videos)} cenas unidas em lei_de_coulomb_completo.mp4")

try:
    from google.colab import files
    files.download("lei_de_coulomb_completo.mp4")
except ImportError:
    pass